# PM-GNN：全局结构描述符 + 渗流专用路径描述符

本 notebook 直接基于用户上传的 `PM-GNN_final(2).ipynb` 修改。

所有新增渗流特征均由 PNM 的孔隙坐标、孔喉半径和孔喉长度计算，不使用目标标签。


## 0. 环境说明

建议在你的 TensorFlow 环境中安装以下包：

```bash
pip install tensorflow numpy pandas matplotlib scikit-learn openpyxl networkx tqdm joblib shap
```

如果你使用 Anaconda，也可以在已有 TensorFlow 环境中执行：

```bash
conda install pandas matplotlib scikit-learn openpyxl networkx tqdm joblib
pip install shap
```

**重要：如果前一个版本已经触发过 GPU OOM，请先重启内核，再从第 0 节开始重新运行。**

第 4 步会自动在数据集目录下生成 `graph_cache` 文件夹。第一次构图较慢，后续可直接读取缓存。


In [ ]:
# ============================================================
# 0. 导入依赖
# ============================================================

import os

# 重要：这两行必须在 import tensorflow 之前设置。
# 作用：降低 TensorFlow 一次性占满显存和显存碎片导致的 OOM 风险。
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

# 4070Ti 12G 通常不需要强制 CPU。
# 如果反复运行后仍然 OOM，先重启内核；最后才把下面改成 True。
# True 会强制使用 CPU 训练，速度会慢一些，但可以绕开 GPU OOM。
FORCE_CPU_TRAINING = False
if FORCE_CPU_TRAINING:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import math
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# GPU 显存按需增长：避免 TensorFlow 启动时一次性占满显存。
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print("GPU memory_growth 设置未生效：需要重启内核后从头运行。", e)

# Figure style: use English text and Times New Roman throughout all figures.
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['axes.unicode_minus'] = False


def export_figure_data(data, figure_path, suffix='_data.csv'):
    """Save the data needed to redraw a figure next to that figure.

    Data are stored in a `figure_data` subfolder under the figure output directory.
    `figure_path` can be the corresponding PNG/SVG/PDF path; only its parent and stem are used.
    """
    figure_path = Path(figure_path)
    data_dir = figure_path.parent / 'figure_data'
    data_dir.mkdir(parents=True, exist_ok=True)
    suffix = suffix if str(suffix).startswith('_') else '_' + str(suffix)
    data_path = data_dir / f"{figure_path.stem}{suffix}"

    if isinstance(data, pd.DataFrame):
        df = data.copy()
    elif isinstance(data, dict):
        df = pd.DataFrame(data)
    else:
        df = pd.DataFrame(data)

    df.to_csv(data_path, index=False, encoding='utf-8-sig')
    return data_path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow 版本:', tf.__version__)
print('GPU 设备:', tf.config.list_physical_devices('GPU'))
print('FORCE_CPU_TRAINING:', FORCE_CPU_TRAINING)


## 1. 路径设置

这里已改为适配 `data_make_DRP317_11sandstone_big_dataset_v3.ipynb` 生成的 11 个 DRP-317 砂岩样品大数据集。默认读取：

```text
H:\Deepl_Project\GNN\ROCK_data\data\DRP317_11sandstone_GNN_dataset_v3\summary\DRP317_11sandstone_GNN_dataset_v2_master.csv
```

如果你的数据制作结果输出目录不同，只需要修改下面的 `DATA_ROOT` 或 `DATASET_PREFIX`。


In [ ]:
# ============================================================
# 1. 路径设置：PM-GNN 渗流精度优化
# ============================================================

# 这里应指向你存放 11 个砂岩样品以及数据制作结果的总目录。
# 数据制作 notebook 会在该目录下生成：DRP317_11sandstone_GNN_dataset_v3
# GitHub portable configuration (only I/O paths changed; modelling code retained).
# Launch Jupyter from the repository root, or set PMGNN_PROJECT_ROOT first.
PROJECT_ROOT = Path(os.environ.get("PMGNN_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("PMGNN_DATA_ROOT", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATASET_PREFIX = "DRP317_11sandstone"

# data_make_DRP317_11sandstone_big_dataset_v3.ipynb 的统一输出目录。
BASE_DIR = DATA_ROOT / f"{DATASET_PREFIX}_GNN_dataset_v3"
PORE_DIR = BASE_DIR / "pore"
PERM_DIR = BASE_DIR / "Permeation"
MECH_DIR = BASE_DIR / "Mechanics"
SUMMARY_DIR = BASE_DIR / "summary"

# 11 样品大数据集总表。
# 注意：数据制作 notebook 版本是 v3，但总表文件名沿用 v2_master，表示表结构版本。
master_csv_path = SUMMARY_DIR / f"{DATASET_PREFIX}_GNN_dataset_v2_master.csv"
master_xlsx_path = SUMMARY_DIR / f"{DATASET_PREFIX}_GNN_dataset_v2_master.xlsx"

# 如果手动移动过数据集，自动在 DATA_ROOT 下递归寻找 master 表。
if not master_csv_path.exists() and not master_xlsx_path.exists() and DATA_ROOT.exists():
    candidates = sorted(DATA_ROOT.rglob(f"{DATASET_PREFIX}_GNN_dataset_v2_master.csv"))
    if candidates:
        master_csv_path = candidates[0]
        SUMMARY_DIR = master_csv_path.parent
        BASE_DIR = SUMMARY_DIR.parent
        PORE_DIR = BASE_DIR / "pore"
        PERM_DIR = BASE_DIR / "Permeation"
        MECH_DIR = BASE_DIR / "Mechanics"
        master_xlsx_path = SUMMARY_DIR / f"{DATASET_PREFIX}_GNN_dataset_v2_master.xlsx"
        print("自动定位到 master 总表:", master_csv_path)

# 如果 master 总表还没生成，备用读取三个 v2 汇总表并合并。
pore_summary_path = SUMMARY_DIR / f"{DATASET_PREFIX}_pore_summary_v2.csv"
perm_summary_path = SUMMARY_DIR / f"{DATASET_PREFIX}_permeation_summary_v2.csv"
mech_summary_path = SUMMARY_DIR / f"{DATASET_PREFIX}_mechanics_summary_v2.csv"

# 本次训练输出目录：最终 PM-GNN（GNN + global_descriptors）。
RESULT_DIR = BASE_DIR / "Final-PM-GNN"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# 当前基础版采用岩样级分组划分，默认 9 个岩样训练 + 2 个岩样验证。
# 后续第7节会根据实际识别到的岩样数量更新 SPLIT_MODE 字符串。
SPLIT_MODE = "rock_group_9train_2val"

# v2/v3 数据中每个样本有 x/y/z 三个方向 PNM，这里选择一个主图给当前 GNN 使用。
# 优先使用第一个贯通方向的 PNM；如果没有贯通方向，该样本会在构图阶段被跳过。
DIRECTION_ORDER = ["x", "y", "z"]

print("11样品数据根目录:", DATA_ROOT)
print("训练读取数据集目录:", BASE_DIR)
print("summary 目录:", SUMMARY_DIR)
print("pore 目录:", PORE_DIR)
print("训练结果输出目录:", RESULT_DIR)
print("master CSV:", master_csv_path)
print("数据划分模式:", SPLIT_MODE)



## 2. 读取 11 样品大数据集总表并准备训练索引

这一部分已改为优先读取 `DRP317_11sandstone_GNN_dataset_v2_master.csv`。如果总表不存在，则自动读取并合并 `pore/permeation/mechanics` 三个汇总表。

注意：该数据集中每个样本可能具有 `pnm_x_xlsx / pnm_y_xlsx / pnm_z_xlsx` 三个方向的孔隙网络。当前训练代码会自动选择一个主 PNM 图作为 GNN 输入，并保留 `rock_id / rock_name` 等岩样来源信息。


In [ ]:
# ============================================================
# 2. 读取 11 样品大数据集总表，并生成当前 GNN 使用的主 PNM 路径
# ============================================================

def read_csv_or_excel(csv_path, xlsx_path=None):
    if Path(csv_path).exists():
        return pd.read_csv(csv_path)
    if xlsx_path is not None and Path(xlsx_path).exists():
        return pd.read_excel(xlsx_path)
    raise FileNotFoundError(f"找不到数据表: {csv_path}")


def read_master_table():
    """优先读取 data_make_DRP317_11sandstone_big_dataset_v3.ipynb 生成的最终总表。"""
    if master_csv_path.exists() or master_xlsx_path.exists():
        print("读取 11 样品 master 总表。")
        return read_csv_or_excel(master_csv_path, master_xlsx_path)

    print("未找到 master 总表，尝试读取并合并三个汇总表。")
    missing = [p for p in [pore_summary_path, perm_summary_path, mech_summary_path] if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "未找到 master 总表，且以下汇总表也缺失。请先运行数据制作 notebook 完成数据生成：\n" +
            "\n".join(str(p) for p in missing)
        )

    pore_df = pd.read_csv(pore_summary_path)
    perm_df = pd.read_csv(perm_summary_path)
    mech_df = pd.read_csv(mech_summary_path)

    # sample_id 在新数据制作流程中已经包含 rock_id 与空间块编号，足以唯一定位样本。
    merge_keys = [c for c in ["sample_id", "z_start", "y_start", "x_start", "block_size"] if c in pore_df.columns]
    all_df = pore_df.merge(
        perm_df,
        on=merge_keys,
        how="left",
        suffixes=("", "_perm")
    ).merge(
        mech_df,
        on=merge_keys,
        how="left",
        suffixes=("", "_mech")
    )
    return all_df


def is_valid_path_value(value):
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    text = str(value).strip()
    return text != "" and text.lower() != "nan"


def filename_from_any_path(path_value):
    """同时兼容 Windows 反斜杠路径和普通路径，取出文件名。"""
    text = str(path_value).replace("\\", "/")
    return text.split("/")[-1]


def safe_int_value(value, default=0):
    val = pd.to_numeric(value, errors="coerce")
    if pd.isna(val):
        return default
    return int(val)


def resolve_existing_path(path_value, fallback_dir=None):
    """
    兼容两种情况：
    1. 总表中保存的是完整绝对路径；
    2. 数据集移动后，绝对路径失效，此时用文件名在当前 PORE_DIR / PERM_DIR / MECH_DIR 下重建路径。
    """
    if not is_valid_path_value(path_value):
        return ""

    p = Path(str(path_value))
    if p.exists():
        return str(p)

    if fallback_dir is not None:
        fallback = Path(fallback_dir) / filename_from_any_path(path_value)
        if fallback.exists():
            return str(fallback)
        return str(fallback)

    return str(p)


def select_primary_pnm(row):
    """
    每个样本有 x/y/z 三个方向性 PNM。
    当前 GNN 使用一个主图输入：优先选第一个贯通方向；若均不贯通则跳过。
    """
    # 兼容旧版单 PNM 字段，但必须避免把空值误判为有效路径。
    if "pnm_xlsx" in row.index and is_valid_path_value(row.get("pnm_xlsx")):
        return resolve_existing_path(row.get("pnm_xlsx"), PORE_DIR), "legacy"

    for d in DIRECTION_ORDER:
        pnm_col = f"pnm_{d}_xlsx"
        per_col = f"percolates_{d}"
        if pnm_col not in row.index:
            continue
        per_flag = safe_int_value(row.get(per_col, 0), default=0)
        if per_flag == 1 and is_valid_path_value(row.get(pnm_col)):
            return resolve_existing_path(row.get(pnm_col), PORE_DIR), d

    return "", "none"


all_df = read_master_table()
all_df = all_df.replace([np.inf, -np.inf], np.nan)

# 兼容 connected_porosity：若缺失，就从 x/y/z 方向连通孔隙率取最大值。
if "connected_porosity" not in all_df.columns:
    cands = [c for c in ["connected_porosity_x", "connected_porosity_y", "connected_porosity_z"] if c in all_df.columns]
    if cands:
        all_df["connected_porosity"] = all_df[cands].max(axis=1)

# 若没有 rock_id，但 sample_id 中含有岩样前缀，则尝试自动恢复。
if "rock_id" not in all_df.columns and "sample_id" in all_df.columns:
    all_df["rock_id"] = all_df["sample_id"].astype(str).str.replace(r"_\d{4}_z.*$", "", regex=True)
if "rock_name" not in all_df.columns:
    all_df["rock_name"] = all_df.get("rock_id", "unknown")

primary_paths = []
primary_dirs = []
for _, row in all_df.iterrows():
    p, d = select_primary_pnm(row)
    primary_paths.append(p)
    primary_dirs.append(d)

# 保留 pnm_xlsx 这个兼容字段，后面的 GNN 构图函数不用再关心新旧数据差异。
all_df["primary_pnm_xlsx"] = primary_paths
all_df["primary_pnm_direction"] = primary_dirs
all_df["pnm_xlsx"] = all_df["primary_pnm_xlsx"]

# 关键字段检查
required_cols = [
    "sample_id", "rock_id", "porosity", "connected_porosity",
    "kx_um2", "ky_um2", "kz_um2",
    "Ex_spring_GPa", "Ey_spring_GPa", "Ez_spring_GPa",
    "pnm_xlsx"
]
missing_cols = [c for c in required_cols if c not in all_df.columns]
if missing_cols:
    raise KeyError(f"当前数据表缺少训练必需字段: {missing_cols}")

print("合并后的样本数量:", len(all_df))
print("岩样数量:", all_df["rock_id"].nunique())
print("各岩样样本数:")
print(all_df["rock_id"].value_counts(dropna=False))
print("\n主 PNM 方向统计:")
print(all_df["primary_pnm_direction"].value_counts(dropna=False))

display(all_df.head())



## 3. 数据质量检查与中文可视化

先检查样本状态、孔隙率、孔喉数量、渗透率和力学代理参数分布。这里的目标不是训练，而是先判断数据是否异常。


In [ ]:
# ============================================================
# 3. 数据质量检查
# ============================================================

for status_col, title in [
    ("pore_status", "孔隙处理状态"),
    ("permeation_status", "渗流计算状态"),
    ("mechanics_status", "力学计算状态"),
]:
    if status_col in all_df.columns:
        print(f"{title}:")
        print(all_df[status_col].value_counts(dropna=False))
        print()

if all(c in all_df.columns for c in ["percolates_x", "percolates_y", "percolates_z"]):
    print("方向性贯通比例:")
    for d in ["x", "y", "z"]:
        print(f"percolates_{d}: {pd.to_numeric(all_df[f'percolates_{d}'], errors='coerce').mean():.4f}")
    print()

label_cols = [
    "kx_um2", "ky_um2", "kz_um2",
    "Ex_spring_GPa", "Ey_spring_GPa", "Ez_spring_GPa"
]

clean_df = all_df.copy()

# 状态字段存在时才过滤；兼容旧表或手动整理后的表。
for status_col in ["pore_status", "permeation_status", "mechanics_status"]:
    if status_col in clean_df.columns:
        clean_df = clean_df[clean_df[status_col].astype(str).str.startswith("ok")]

# 当前 GNN 先使用一个主 PNM 图跑通，因此没有贯通 PNM 的样本先跳过。
clean_df = clean_df[clean_df["primary_pnm_direction"].isin(DIRECTION_ORDER + ["legacy"])]
clean_df = clean_df[clean_df["pnm_xlsx"].astype(str).str.len() > 0]

# 标签必须完整且为有限值。
for col in label_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")
clean_df = clean_df.replace([np.inf, -np.inf], np.nan)
clean_df = clean_df.dropna(subset=label_cols + ["pnm_xlsx", "porosity", "connected_porosity"])

print("清洗后的样本数量:", len(clean_df))
if "rock_id" in clean_df.columns:
    print("清洗后岩样数量:", clean_df["rock_id"].nunique())
    print("清洗后各岩样样本数:")
    print(clean_df["rock_id"].value_counts(dropna=False))
print("清洗后主 PNM 方向统计:")
print(clean_df["primary_pnm_direction"].value_counts(dropna=False))

summary_cols = label_cols + ["porosity", "connected_porosity"]
for extra_col in ["num_pores", "num_throats", "x_num_pores", "y_num_pores", "z_num_pores"]:
    if extra_col in clean_df.columns:
        summary_cols.append(extra_col)

display(clean_df[summary_cols].describe())


In [ ]:
# ============================================================
# 3.1 Distributions of key variables
# ============================================================

plot_cols = [
    ("porosity", "Porosity"),
    ("connected_porosity", "Connected porosity"),
    ("num_pores", "Number of pores in the primary graph"),
    ("num_throats", "Number of throats in the primary graph"),
    ("kx_um2", "X-direction permeability"),
    ("ky_um2", "Y-direction permeability"),
    ("kz_um2", "Z-direction permeability"),
    ("Ex_spring_GPa", "X-direction equivalent elastic modulus"),
    ("Ey_spring_GPa", "Y-direction equivalent elastic modulus"),
    ("Ez_spring_GPa", "Z-direction equivalent elastic modulus")
]

for col, cname in plot_cols:
    if col not in clean_df.columns:
        continue
    values = pd.to_numeric(clean_df[col], errors="coerce").dropna().values
    if len(values) == 0:
        continue

    bins = min(30, max(5, len(values)))
    counts, bin_edges = np.histogram(values, bins=bins)
    fig_path = RESULT_DIR / f'key_variable_distribution_{col}.png'
    export_figure_data(pd.DataFrame({col: values}), fig_path, suffix='_raw_values.csv')
    export_figure_data(
        pd.DataFrame({
            'bin_left': bin_edges[:-1],
            'bin_right': bin_edges[1:],
            'count': counts
        }),
        fig_path,
        suffix='_histogram_bins.csv'
    )

    plt.figure(figsize=(6, 4))
    plt.hist(values, bins=bins)
    plt.xlabel(cname)
    plt.ylabel("Number of samples")
    plt.title(f"Distribution of {cname}")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()


## 4. 从 PNM Excel 构建图数据，并生成 global_descriptors


In [ ]:

# ============================================================
# 4. 读取 v2 PNM Excel，构建图数据
# ============================================================

NODE_FEATURE_COLS = [
    "Volume [cm^3]",
    "Area [cm^2]",
    "EqRadius [cm]",
    "X Coord [cm]",
    "Y Coord [cm]",
    "Z Coord [cm]",
    "Coordination Number"
]

EDGE_FEATURE_COLS = [
    "Area [cm^2]",
    "EqRadius [cm]",
    "ChannelLength [cm]"
]

TARGET_COLS = [
    "kx_um2", "ky_um2", "kz_um2",
    "Ex_spring_GPa", "Ey_spring_GPa", "Ez_spring_GPa"
]

# ============================================================
# 新增：图级全局结构特征
# 说明：
# 当前结果中渗流 R² 偏低、力学 R² 一般，一个重要原因是模型只用节点平均池化，
# 很难稳定捕捉孔隙率、连通孔隙率、孔喉尺度分布、方向贯通等图级结构信息。
# 这里将样本级结构特征与 PNM 统计特征加入 GNN 输入，每个节点重复拼接同一组全局特征。
# ============================================================

ROW_GLOBAL_FEATURE_CANDIDATES = [
    # 孔隙/连通结构
    "porosity", "connected_porosity", "num_pores", "num_throats",
    "pore_volume", "pore_area", "pore_surface_area_density",
    "pore_volume_density", "throat_length_density",

    # 三方向孔隙/渗流状态
    "percolates_x", "percolates_y", "percolates_z",
    "connected_porosity_x", "connected_porosity_y", "connected_porosity_z",
    "num_pores_x", "num_pores_y", "num_pores_z",
    "num_throats_x", "num_throats_y", "num_throats_z",

    # 固体骨架/力学相关结构信息（如果数据集中存在则自动启用）
    "solid_fraction",
    "solid_connected_fraction_x", "solid_connected_fraction_y", "solid_connected_fraction_z",
    "solid_skeleton_fraction",
    "solid_skeleton_length_density",
    "solid_skeleton_percolates_x", "solid_skeleton_percolates_y", "solid_skeleton_percolates_z",
    "solid_skeleton_percolating_components_x",
    "solid_skeleton_percolating_components_y",
    "solid_skeleton_percolating_components_z",
]

_table_for_global_features = clean_df if "clean_df" in globals() else all_df
ROW_GLOBAL_FEATURE_COLS = [
    c for c in ROW_GLOBAL_FEATURE_CANDIDATES
    if c in _table_for_global_features.columns and c not in TARGET_COLS
]


# ============================================================
# 渗流专用路径描述符
# ============================================================
# 这些特征全部由 PNM 的孔隙坐标、孔喉半径和孔喉长度计算，
# 不使用 kx/ky/kz 标签，因此不会造成标签泄漏。
#
# 关键思想：
# 1. 以 r^4/L 作为孔喉水力导通能力代理；
# 2. 提取 x/y/z 三方向的贯通骨架、最优路径、瓶颈和迂曲度；
# 3. 将路径级信息作为 global_descriptors 的一部分输入渗流模型。

FLOW_BOUNDARY_FRACTION = 0.05
FLOW_LOG_FLOOR = -30.0

FLOW_GLOBAL_FEATURE_COLS = [
    "flow_dead_end_fraction",
    "flow_log10_conductance_p10",
    "flow_log10_conductance_p50",
    "flow_log10_conductance_p90",

    "flow_x_path_exists",
    "flow_x_backbone_node_fraction",
    "flow_x_tortuosity",
    "flow_x_log10_best_path_conductance",
    "flow_x_log10_widest_bottleneck",
    "flow_x_log10_directional_conductance_sum",

    "flow_y_path_exists",
    "flow_y_backbone_node_fraction",
    "flow_y_tortuosity",
    "flow_y_log10_best_path_conductance",
    "flow_y_log10_widest_bottleneck",
    "flow_y_log10_directional_conductance_sum",

    "flow_z_path_exists",
    "flow_z_backbone_node_fraction",
    "flow_z_tortuosity",
    "flow_z_log10_best_path_conductance",
    "flow_z_log10_widest_bottleneck",
    "flow_z_log10_directional_conductance_sum",
]


def _flow_safe_log10(value, floor=FLOW_LOG_FLOOR):
    value = float(value)
    if not np.isfinite(value) or value <= 0:
        return float(floor)
    return float(max(np.log10(value), floor))


def _flow_boundary_nodes(values, fraction=FLOW_BOUNDARY_FRACTION):
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64)

    order = np.argsort(values)
    k = max(1, int(np.ceil(n * fraction)))
    # 避免入口和出口边界覆盖过多节点。
    k = min(k, max(1, n // 4))

    inlet = order[:k].astype(np.int64)
    outlet = order[-k:].astype(np.int64)

    # 极小图中避免入口、出口完全重合。
    if n > 1:
        outlet = np.asarray(
            [v for v in outlet if v not in set(inlet.tolist())],
            dtype=np.int64
        )
        if len(outlet) == 0:
            outlet = np.asarray([order[-1]], dtype=np.int64)

    return inlet, outlet


def _flow_union_find_labels(num_nodes, src, dst):
    parent = np.arange(num_nodes, dtype=np.int64)
    rank = np.zeros(num_nodes, dtype=np.int8)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        ra = find(int(a))
        rb = find(int(b))
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    for u, v in zip(src, dst):
        if u != v:
            union(u, v)

    return np.asarray([find(i) for i in range(num_nodes)], dtype=np.int64)


def _flow_multisource_shortest(adj, sources, targets, weight_pos):
    import heapq

    if len(sources) == 0 or len(targets) == 0:
        return np.inf

    target_set = set(int(v) for v in targets)
    dist = np.full(len(adj), np.inf, dtype=float)
    heap = []

    for s in sources:
        s = int(s)
        dist[s] = 0.0
        heapq.heappush(heap, (0.0, s))

    while heap:
        d_u, u = heapq.heappop(heap)
        if d_u != dist[u]:
            continue
        if u in target_set:
            return float(d_u)

        for item in adj[u]:
            v = item[0]
            w = float(item[weight_pos])
            if not np.isfinite(w) or w < 0:
                continue
            nd = d_u + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(heap, (nd, v))

    return np.inf


def _flow_multisource_widest(adj, sources, targets):
    import heapq

    if len(sources) == 0 or len(targets) == 0:
        return 0.0

    target_set = set(int(v) for v in targets)
    score = np.zeros(len(adj), dtype=float)
    heap = []

    for s in sources:
        s = int(s)
        score[s] = np.inf
        heapq.heappush(heap, (-score[s], s))

    while heap:
        neg_score, u = heapq.heappop(heap)
        current = -neg_score
        if current < score[u]:
            continue
        if u in target_set:
            return float(current)

        for item in adj[u]:
            v = item[0]
            conductance = float(item[3])
            candidate = min(current, conductance)
            if candidate > score[v]:
                score[v] = candidate
                heapq.heappush(heap, (-candidate, v))

    return 0.0


def compute_flow_path_descriptors(pores, throats):
    """
    计算 22 个渗流专用 global_descriptors。

    返回的路径特征包括：
    - 孔喉水力导通能力 r^4/L 的分位数；
    - x/y/z 方向是否存在贯通路径；
    - 贯通骨架节点比例；
    - 几何最短路径迂曲度；
    - 最低总阻力路径对应的等效导通能力；
    - 最宽路径的瓶颈导通能力；
    - 方向投影后的总导通能力。
    """
    result = {name: 0.0 for name in FLOW_GLOBAL_FEATURE_COLS}

    required_pore_cols = [
        "X Coord [cm]", "Y Coord [cm]", "Z Coord [cm]"
    ]
    required_throat_cols = [
        "Pore ID #1", "Pore ID #2",
        "EqRadius [cm]", "ChannelLength [cm]"
    ]

    if any(c not in pores.columns for c in required_pore_cols):
        return result
    if any(c not in throats.columns for c in required_throat_cols):
        return result

    coords = (
        pores[required_pore_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=float)
    )
    n_nodes = len(coords)
    if n_nodes < 2:
        return result

    src = pd.to_numeric(
        throats["Pore ID #1"], errors="coerce"
    ).fillna(-1).to_numpy(dtype=np.int64)
    dst = pd.to_numeric(
        throats["Pore ID #2"], errors="coerce"
    ).fillna(-1).to_numpy(dtype=np.int64)
    radius = pd.to_numeric(
        throats["EqRadius [cm]"], errors="coerce"
    ).fillna(0.0).to_numpy(dtype=float)
    channel_length = pd.to_numeric(
        throats["ChannelLength [cm]"], errors="coerce"
    ).fillna(0.0).to_numpy(dtype=float)

    valid = (
        (src >= 0) & (src < n_nodes) &
        (dst >= 0) & (dst < n_nodes) &
        (src != dst)
    )
    src = src[valid]
    dst = dst[valid]
    radius = radius[valid]
    channel_length = channel_length[valid]

    if len(src) == 0:
        return result

    delta = coords[dst] - coords[src]
    geometric_length = np.linalg.norm(delta, axis=1)

    eps = 1e-18
    effective_length = np.where(
        channel_length > eps,
        channel_length,
        geometric_length
    )
    effective_length = np.maximum(effective_length, eps)
    radius = np.maximum(radius, 0.0)

    # Hagen-Poiseuille 型水力导通能力代理。
    conductance = np.power(radius, 4) / effective_length
    conductance = np.nan_to_num(
        conductance,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )
    resistance = 1.0 / np.maximum(conductance, eps)

    positive_g = conductance[conductance > 0]
    if len(positive_g) > 0:
        q10, q50, q90 = np.quantile(
            positive_g, [0.10, 0.50, 0.90]
        )
        result["flow_log10_conductance_p10"] = _flow_safe_log10(q10)
        result["flow_log10_conductance_p50"] = _flow_safe_log10(q50)
        result["flow_log10_conductance_p90"] = _flow_safe_log10(q90)
    else:
        result["flow_log10_conductance_p10"] = FLOW_LOG_FLOOR
        result["flow_log10_conductance_p50"] = FLOW_LOG_FLOOR
        result["flow_log10_conductance_p90"] = FLOW_LOG_FLOOR

    degree = np.zeros(n_nodes, dtype=np.int64)
    np.add.at(degree, src, 1)
    np.add.at(degree, dst, 1)
    result["flow_dead_end_fraction"] = float(
        np.mean(degree <= 1)
    )

    labels = _flow_union_find_labels(
        n_nodes, src, dst
    )

    # 邻接表元素：(邻居, 几何长度, 水力阻力, 水力导通能力)
    adj = [[] for _ in range(n_nodes)]
    for u, v, gl, r_e, g_e in zip(
        src, dst, geometric_length, resistance, conductance
    ):
        u = int(u)
        v = int(v)
        gl = float(max(gl, eps))
        r_e = float(max(r_e, eps))
        g_e = float(max(g_e, 0.0))
        adj[u].append((v, gl, r_e, g_e))
        adj[v].append((u, gl, r_e, g_e))

    safe_geom = np.maximum(geometric_length, eps)
    direction_cos_sq = np.square(delta / safe_geom[:, None])

    axis_names = ["x", "y", "z"]
    for axis, axis_name in enumerate(axis_names):
        span = float(
            np.max(coords[:, axis]) -
            np.min(coords[:, axis])
        )

        if span <= eps:
            result[f"flow_{axis_name}_path_exists"] = 0.0
            result[
                f"flow_{axis_name}_backbone_node_fraction"
            ] = 0.0
            result[f"flow_{axis_name}_tortuosity"] = 0.0
            result[
                f"flow_{axis_name}_log10_best_path_conductance"
            ] = FLOW_LOG_FLOOR
            result[
                f"flow_{axis_name}_log10_widest_bottleneck"
            ] = FLOW_LOG_FLOOR
            result[
                f"flow_{axis_name}_log10_directional_conductance_sum"
            ] = FLOW_LOG_FLOOR
            continue

        inlet, outlet = _flow_boundary_nodes(
            coords[:, axis]
        )

        inlet_labels = set(labels[inlet].tolist())
        outlet_labels = set(labels[outlet].tolist())
        percolating_labels = inlet_labels.intersection(
            outlet_labels
        )

        path_exists = len(percolating_labels) > 0
        result[f"flow_{axis_name}_path_exists"] = float(path_exists)

        if path_exists:
            backbone_mask = np.isin(
                labels,
                list(percolating_labels)
            )
            result[
                f"flow_{axis_name}_backbone_node_fraction"
            ] = float(np.mean(backbone_mask))
        else:
            result[
                f"flow_{axis_name}_backbone_node_fraction"
            ] = 0.0

        shortest_geom = _flow_multisource_shortest(
            adj, inlet, outlet, weight_pos=1
        )
        if np.isfinite(shortest_geom) and span > eps:
            result[f"flow_{axis_name}_tortuosity"] = float(
                max(shortest_geom / span, 1.0)
            )
        else:
            result[f"flow_{axis_name}_tortuosity"] = 0.0

        shortest_resistance = _flow_multisource_shortest(
            adj, inlet, outlet, weight_pos=2
        )
        best_path_conductance = (
            1.0 / shortest_resistance
            if np.isfinite(shortest_resistance)
            and shortest_resistance > 0
            else 0.0
        )
        result[
            f"flow_{axis_name}_log10_best_path_conductance"
        ] = _flow_safe_log10(best_path_conductance)

        widest_bottleneck = _flow_multisource_widest(
            adj, inlet, outlet
        )
        result[
            f"flow_{axis_name}_log10_widest_bottleneck"
        ] = _flow_safe_log10(widest_bottleneck)

        directional_sum = float(
            np.sum(conductance * direction_cos_sq[:, axis])
        )
        result[
            f"flow_{axis_name}_log10_directional_conductance_sum"
        ] = _flow_safe_log10(directional_sum)

    return result


COMPUTED_GLOBAL_FEATURE_COLS = [
    "graph_num_pores",
    "graph_num_throats",
    "graph_edge_density",
    "graph_mean_degree",
    "graph_std_degree",
    "graph_max_degree",
    "graph_mean_pore_radius",
    "graph_std_pore_radius",
    "graph_max_pore_radius",
    "graph_mean_throat_radius",
    "graph_std_throat_radius",
    "graph_max_throat_radius",
    "graph_mean_throat_length",
    "graph_std_throat_length",
    "graph_max_throat_length",
    "graph_bbox_x",
    "graph_bbox_y",
    "graph_bbox_z",
    "graph_bbox_volume",
    "primary_dir_is_x",
    "primary_dir_is_y",
    "primary_dir_is_z",
] + FLOW_GLOBAL_FEATURE_COLS

BASE_GLOBAL_FEATURE_COLS = [
    c for c in (ROW_GLOBAL_FEATURE_COLS + COMPUTED_GLOBAL_FEATURE_COLS)
    if c not in FLOW_GLOBAL_FEATURE_COLS
]
GLOBAL_FEATURE_COLS = ROW_GLOBAL_FEATURE_COLS + COMPUTED_GLOBAL_FEATURE_COLS

print("启用的样本级全局特征数量:", len(ROW_GLOBAL_FEATURE_COLS))
print("启用的PNM统计与渗流路径全局特征数量:", len(COMPUTED_GLOBAL_FEATURE_COLS))
print("其中渗流专用路径描述符数量:", len(FLOW_GLOBAL_FEATURE_COLS))
print("GNN节点输入将拼接全局结构特征总数:", len(GLOBAL_FEATURE_COLS))


def safe_float_from_row(row, col, default=np.nan):
    if col not in row.index:
        return default
    val = pd.to_numeric(row.get(col), errors="coerce")
    if pd.isna(val):
        return default
    return float(val)


def _safe_array_stats(values):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().values
    if len(arr) == 0:
        return 0.0, 0.0, 0.0
    return float(np.mean(arr)), float(np.std(arr)), float(np.max(arr))


def _safe_range(values):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().values
    if len(arr) == 0:
        return 0.0
    return float(np.max(arr) - np.min(arr))


def read_pnm_graph(row):
    """读取单个样本的主 PNM Excel，返回节点特征、边索引、边特征、图级特征和标签。"""
    pnm_path = Path(row["pnm_xlsx"])
    if not pnm_path.exists():
        # 数据集移动后，尝试用文件名在当前 PORE_DIR 下寻找。
        pnm_path = PORE_DIR / filename_from_any_path(row["pnm_xlsx"])
    if not pnm_path.exists():
        return None

    try:
        pores = pd.read_excel(pnm_path, sheet_name="Pores")
        throats = pd.read_excel(pnm_path, sheet_name="Throats")
    except Exception:
        return None

    if len(pores) == 0 or len(throats) == 0:
        return None

    missing_node_cols = [c for c in NODE_FEATURE_COLS if c not in pores.columns]
    missing_edge_cols = [c for c in EDGE_FEATURE_COLS if c not in throats.columns]
    if missing_node_cols or missing_edge_cols:
        return None

    # 节点特征
    x = pores[NODE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)

    # 边索引，构建无向图，所以每条孔喉复制为两个方向。
    src = pd.to_numeric(throats["Pore ID #1"], errors="coerce").fillna(-1).values.astype(np.int64)
    dst = pd.to_numeric(throats["Pore ID #2"], errors="coerce").fillna(-1).values.astype(np.int64)
    edge_index = np.vstack([np.concatenate([src, dst]), np.concatenate([dst, src])]).T

    # 边特征也复制为两个方向。
    e = throats[EDGE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
    edge_attr = np.concatenate([e, e], axis=0)

    # 过滤越界边。
    n = x.shape[0]
    valid = (
        (edge_index[:, 0] >= 0) & (edge_index[:, 0] < n) &
        (edge_index[:, 1] >= 0) & (edge_index[:, 1] < n)
    )
    edge_index = edge_index[valid]
    edge_attr = edge_attr[valid]

    if len(edge_index) == 0:
        return None

    y = pd.to_numeric(row[TARGET_COLS], errors="coerce").values.astype(np.float32)
    if not np.all(np.isfinite(y)):
        return None

    # 图级拓扑统计属性：供第13部分 GNN 图级嵌入可视化使用。
    # 注意：edge_index 已经是双向边，因此孔喉数量需要除以 2。
    num_pores_calc = int(x.shape[0])
    num_throats_calc = int(len(edge_index) // 2)

    pore_radius = pd.to_numeric(pores["EqRadius [cm]"], errors="coerce") if "EqRadius [cm]" in pores.columns else pd.Series(dtype=float)
    throat_radius = pd.to_numeric(throats["EqRadius [cm]"], errors="coerce") if "EqRadius [cm]" in throats.columns else pd.Series(dtype=float)
    throat_length = pd.to_numeric(throats["ChannelLength [cm]"], errors="coerce") if "ChannelLength [cm]" in throats.columns else pd.Series(dtype=float)
    coord_num = pd.to_numeric(pores["Coordination Number"], errors="coerce") if "Coordination Number" in pores.columns else pd.Series(dtype=float)

    avg_pore_radius = float(pore_radius.dropna().mean()) if len(pore_radius.dropna()) > 0 else np.nan
    avg_throat_radius = float(throat_radius.dropna().mean()) if len(throat_radius.dropna()) > 0 else np.nan
    coordination_number = float(coord_num.dropna().mean()) if len(coord_num.dropna()) > 0 else np.nan

    num_pores_value = safe_float_from_row(row, "num_pores", num_pores_calc)
    num_throats_value = safe_float_from_row(row, "num_throats", num_throats_calc)

    # 计算 GNN 可直接利用的图级全局结构特征。
    degree = np.bincount(edge_index[:, 0], minlength=num_pores_calc).astype(float)
    mean_degree, std_degree, max_degree = _safe_array_stats(degree)
    mean_pore_r, std_pore_r, max_pore_r = _safe_array_stats(pore_radius)
    mean_throat_r, std_throat_r, max_throat_r = _safe_array_stats(throat_radius)
    mean_throat_l, std_throat_l, max_throat_l = _safe_array_stats(throat_length)

    bbox_x = _safe_range(pores["X Coord [cm]"])
    bbox_y = _safe_range(pores["Y Coord [cm]"])
    bbox_z = _safe_range(pores["Z Coord [cm]"])
    bbox_volume = float(max(bbox_x, 1e-12) * max(bbox_y, 1e-12) * max(bbox_z, 1e-12))

    primary_dir = str(row.get("primary_pnm_direction", "unknown")).lower()

    # 从当前 PNM 中提取 x/y/z 方向的路径级渗流描述符。
    flow_global = compute_flow_path_descriptors(pores, throats)

    computed_global = {
        "graph_num_pores": float(num_pores_calc),
        "graph_num_throats": float(num_throats_calc),
        "graph_edge_density": float(num_throats_calc / max(num_pores_calc * (num_pores_calc - 1) / 2, 1.0)),
        "graph_mean_degree": mean_degree,
        "graph_std_degree": std_degree,
        "graph_max_degree": max_degree,
        "graph_mean_pore_radius": mean_pore_r,
        "graph_std_pore_radius": std_pore_r,
        "graph_max_pore_radius": max_pore_r,
        "graph_mean_throat_radius": mean_throat_r,
        "graph_std_throat_radius": std_throat_r,
        "graph_max_throat_radius": max_throat_r,
        "graph_mean_throat_length": mean_throat_l,
        "graph_std_throat_length": std_throat_l,
        "graph_max_throat_length": max_throat_l,
        "graph_bbox_x": bbox_x,
        "graph_bbox_y": bbox_y,
        "graph_bbox_z": bbox_z,
        "graph_bbox_volume": bbox_volume,
        "primary_dir_is_x": 1.0 if primary_dir == "x" else 0.0,
        "primary_dir_is_y": 1.0 if primary_dir == "y" else 0.0,
        "primary_dir_is_z": 1.0 if primary_dir == "z" else 0.0,
    }
    computed_global.update(flow_global)

    global_values = []
    for col in GLOBAL_FEATURE_COLS:
        if col in computed_global:
            val = computed_global[col]
        else:
            val = safe_float_from_row(row, col, default=0.0)
        if not np.isfinite(val):
            val = 0.0
        global_values.append(float(val))

    graph = {
        "sample_id": row["sample_id"],
        "rock_id": row.get("rock_id", "unknown"),
        "rock_name": row.get("rock_name", row.get("rock_id", "unknown")),
        "global_block_id": row.get("global_block_id", np.nan),
        "x": x,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "global_features": np.asarray(global_values, dtype=np.float32),
        "global_feature_names": GLOBAL_FEATURE_COLS,
        "y": y,
        "y_raw": y.copy(),
        "porosity": safe_float_from_row(row, "porosity"),
        "connected_porosity": safe_float_from_row(row, "connected_porosity"),
        "num_pores": float(num_pores_value),
        "num_throats": float(num_throats_value),
        "avg_pore_radius": avg_pore_radius,
        "avg_throat_radius": avg_throat_radius,
        "coordination_number": coordination_number,
        "pnm_xlsx": str(pnm_path),
        "primary_pnm_direction": row.get("primary_pnm_direction", "unknown"),
        "split_by_x": row.get("split_by_x", ""),
        "split_by_z": row.get("split_by_z", ""),
    }

    for d in ["x", "y", "z"]:
        graph[f"percolates_{d}"] = safe_int_value(row.get(f"percolates_{d}", 0), default=0)

    return graph


# ============================================================
# 图数据缓存：第一次从 Excel 构图，后续直接读取 pkl 缓存
# ============================================================
import pickle
import json
import hashlib
from datetime import datetime
from tqdm import tqdm

GRAPH_CACHE_DIR = BASE_DIR / "graph_cache_flow_path_descriptors"
GRAPH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_CACHE_PATH = GRAPH_CACHE_DIR / f"{DATASET_PREFIX}_pm_gnn_flow_path_graphs.pkl"
GRAPH_CACHE_META_PATH = GRAPH_CACHE_DIR / f"{DATASET_PREFIX}_pm_gnn_flow_path_graphs_meta.json"

# 如需强制重新从 Excel 构图，把这里改为 True。
FORCE_REBUILD_GRAPH_CACHE = False


def _resolve_pnm_path_for_signature(path_value):
    """用于缓存签名：兼容绝对路径失效后按文件名在 PORE_DIR 下寻找。"""
    if not is_valid_path_value(path_value):
        return Path("")
    p = Path(str(path_value))
    if p.exists():
        return p
    fallback = PORE_DIR / filename_from_any_path(path_value)
    return fallback


def _safe_json_value(value):
    """把 numpy/pandas 数值安全转成 JSON 可稳定比较的值。"""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (int, float, str, bool)):
        return value
    return str(value)


def build_graph_cache_signature(df):
    """
    根据当前 clean_df、PNM 文件路径/大小/修改时间、目标字段和特征字段生成签名。
    作用：数据没有变化时直接读缓存；重新生成 Excel 或总表后自动重建缓存。
    """
    sig_rows = []
    check_cols = [
        "sample_id", "rock_id", "pnm_xlsx", "primary_pnm_direction",
        "porosity", "connected_porosity",
    ] + TARGET_COLS + ROW_GLOBAL_FEATURE_COLS
    check_cols = [c for c in check_cols if c in df.columns]

    for _, row in df.iterrows():
        p = _resolve_pnm_path_for_signature(row.get("pnm_xlsx", ""))
        if p.exists():
            try:
                stat = p.stat()
                file_size = int(stat.st_size)
                file_mtime = round(float(stat.st_mtime), 3)
            except OSError:
                file_size = -1
                file_mtime = -1.0
        else:
            file_size = -1
            file_mtime = -1.0

        row_sig = {
            "file_name": filename_from_any_path(row.get("pnm_xlsx", "")),
            "file_size": file_size,
            "file_mtime": file_mtime,
        }
        for col in check_cols:
            row_sig[col] = _safe_json_value(row.get(col))
        sig_rows.append(row_sig)

    payload = {
        "dataset_prefix": DATASET_PREFIX,
        "base_dir": str(BASE_DIR),
        "num_rows": int(len(df)),
        "node_feature_cols": NODE_FEATURE_COLS,
        "edge_feature_cols": EDGE_FEATURE_COLS,
        "target_cols": TARGET_COLS,
        "row_global_feature_cols": ROW_GLOBAL_FEATURE_COLS,
        "computed_global_feature_cols": COMPUTED_GLOBAL_FEATURE_COLS,
        "global_feature_cols": GLOBAL_FEATURE_COLS,
        "flow_descriptor_version": "flow_path_v1",
        "flow_boundary_fraction": FLOW_BOUNDARY_FRACTION,
        "rows": sig_rows,
    }
    text = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    signature = hashlib.md5(text.encode("utf-8")).hexdigest()
    return signature, payload


def load_graph_cache(expected_signature):
    """签名一致时读取图缓存。"""
    if FORCE_REBUILD_GRAPH_CACHE:
        print("FORCE_REBUILD_GRAPH_CACHE=True，强制重新从 Excel 构建图数据。")
        return None, None
    if not GRAPH_CACHE_PATH.exists() or not GRAPH_CACHE_META_PATH.exists():
        return None, None

    try:
        with open(GRAPH_CACHE_META_PATH, "r", encoding="utf-8") as f:
            meta = json.load(f)
    except Exception as e:
        print("缓存元信息读取失败，将重建缓存:", e)
        return None, None

    if meta.get("signature") != expected_signature:
        print("图缓存已存在，但数据签名变化，将自动重建缓存。")
        return None, None

    try:
        with open(GRAPH_CACHE_PATH, "rb") as f:
            cache_obj = pickle.load(f)
        graphs_cached = cache_obj.get("graphs", [])
        failed_cached = cache_obj.get("failed", [])
        print("已直接读取图缓存:", GRAPH_CACHE_PATH)
        print("缓存创建时间:", meta.get("created_at", "unknown"))
        return graphs_cached, failed_cached
    except Exception as e:
        print("图缓存读取失败，将重建缓存:", e)
        return None, None


def save_graph_cache(graphs_to_save, failed_to_save, signature, signature_payload):
    """保存图缓存和元信息。"""
    cache_obj = {
        "graphs": graphs_to_save,
        "failed": failed_to_save,
        "node_feature_cols": NODE_FEATURE_COLS,
        "edge_feature_cols": EDGE_FEATURE_COLS,
        "target_cols": TARGET_COLS,
        "global_feature_cols": GLOBAL_FEATURE_COLS,
    }
    with open(GRAPH_CACHE_PATH, "wb") as f:
        pickle.dump(cache_obj, f, protocol=pickle.HIGHEST_PROTOCOL)

    meta = {
        "signature": signature,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "num_graphs": int(len(graphs_to_save)),
        "num_failed": int(len(failed_to_save)),
        "cache_path": str(GRAPH_CACHE_PATH),
        "signature_payload_summary": {
            "dataset_prefix": signature_payload.get("dataset_prefix"),
            "base_dir": signature_payload.get("base_dir"),
            "num_rows": signature_payload.get("num_rows"),
            "node_feature_cols": signature_payload.get("node_feature_cols"),
            "edge_feature_cols": signature_payload.get("edge_feature_cols"),
            "target_cols": signature_payload.get("target_cols"),
            "global_feature_cols": signature_payload.get("global_feature_cols"),
        },
    }
    with open(GRAPH_CACHE_META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print("图缓存已保存:", GRAPH_CACHE_PATH)
    print("缓存元信息已保存:", GRAPH_CACHE_META_PATH)


cache_signature, cache_signature_payload = build_graph_cache_signature(clean_df)
graphs, failed = load_graph_cache(cache_signature)

if graphs is None:
    graphs = []
    failed = []

    for _, row in tqdm(clean_df.iterrows(), total=len(clean_df), desc="从 PNM Excel 构建图数据"):
        g = read_pnm_graph(row)
        if g is None:
            failed.append(row["sample_id"])
        else:
            graphs.append(g)

    save_graph_cache(graphs, failed, cache_signature, cache_signature_payload)

print("成功构建/读取图数量:", len(graphs))
print("构图失败数量:", len(failed))
if failed[:5]:
    print("前5个失败样本:", failed[:5])

if len(graphs) < 3:
    raise ValueError("可用于训练的 PNM 图少于 3 个，请先检查 v2 数据制作是否完整，尤其是 pnm_x/y/z_xlsx 文件是否存在且非空。")

print("单个节点原始特征维度:", graphs[0]["x"].shape[1])
print("单个图全局结构特征维度:", graphs[0]["global_features"].shape[0])
print("其中渗流路径描述符维度:", len(FLOW_GLOBAL_FEATURE_COLS))
print("第4步提示：后续再次运行时，只要数据未变化，将直接读取 graph_cache 中的 pkl 缓存，不再重复读取 Excel。")


## 5. 孔隙网络图可视化

这一部分改为真实三维可视化：孔隙以三维球体形式表达，孔喉以三维连线形式表达，并按孔喉半径调整线宽。为了避免大网络渲染过慢，默认对孔隙和孔喉进行代表性抽样，但仍会保留整体三维拓扑关系。


In [ ]:

# ============================================================
# 5. 孔隙网络真实三维可视化：3D孔隙球体 + 3D孔喉连线
# ============================================================

def _unique_undirected_edges(edge_index, edge_attr=None):
    """从双向 edge_index 中提取无向唯一孔喉，避免重复绘图。"""
    edge_index = np.asarray(edge_index, dtype=np.int64)
    if edge_index.size == 0:
        if edge_attr is None:
            return edge_index.reshape(0, 2), None
        return edge_index.reshape(0, 2), np.asarray(edge_attr).reshape(0, np.asarray(edge_attr).shape[-1])

    pair_sorted = np.sort(edge_index, axis=1)
    valid = pair_sorted[:, 0] != pair_sorted[:, 1]
    pair_sorted = pair_sorted[valid]
    edge_keep = edge_index[valid]
    attr_keep = None if edge_attr is None else np.asarray(edge_attr)[valid]

    _, keep_idx = np.unique(pair_sorted, axis=0, return_index=True)
    edge_keep = edge_keep[keep_idx]
    if attr_keep is not None:
        attr_keep = attr_keep[keep_idx]
    return edge_keep, attr_keep


def _set_axes_equal_3d(ax, x_coord, y_coord, z_coord):
    """让三维坐标轴等比例显示，避免网络被压扁。"""
    x_min, x_max = np.nanmin(x_coord), np.nanmax(x_coord)
    y_min, y_max = np.nanmin(y_coord), np.nanmax(y_coord)
    z_min, z_max = np.nanmin(z_coord), np.nanmax(z_coord)

    x_mid = 0.5 * (x_min + x_max)
    y_mid = 0.5 * (y_min + y_max)
    z_mid = 0.5 * (z_min + z_max)
    max_range = max(x_max - x_min, y_max - y_min, z_max - z_min, 1e-12)

    ax.set_xlim(x_mid - max_range / 2, x_mid + max_range / 2)
    ax.set_ylim(y_mid - max_range / 2, y_mid + max_range / 2)
    ax.set_zlim(z_mid - max_range / 2, z_mid + max_range / 2)


def _plot_sphere(ax, center, radius, color_value, cmap, norm, alpha=0.70, resolution=10):
    """绘制一个真实三维球体孔隙。"""
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)
    xs = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    ys = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    zs = center[2] + radius * np.outer(np.ones_like(u), np.cos(v))
    color = cmap(norm(color_value))
    ax.plot_surface(xs, ys, zs, color=color, linewidth=0, antialiased=True, shade=True, alpha=alpha)


def visualize_graph_3d(
    graph,
    max_edges=2500,
    max_solid_pores=160,
    save_path=None,
    sphere_resolution=10,
    random_state=SEED
):
    """
    三维显示孔隙—孔喉拓扑网络。

    改进点：
    1. 孔喉不再缺失：用 Line3DCollection 统一绘制无向孔喉；
    2. 孔隙不再只是二维散点：选取代表性孔隙绘制为三维球体；
    3. 孔隙球体半径按 EqRadius 自动放大到可见尺度，保留相对大小差异；
    4. 坐标轴等比例，避免三维网络被压扁成二维效果。
    """
    from mpl_toolkits.mplot3d.art3d import Line3DCollection
    from matplotlib import cm
    from matplotlib.colors import Normalize

    rng = np.random.default_rng(random_state)

    x = np.asarray(graph['x'], dtype=float)
    edge_index = np.asarray(graph['edge_index'], dtype=np.int64)
    edge_attr = np.asarray(graph.get('edge_attr', np.zeros((len(edge_index), 3))), dtype=float)

    x_coord = x[:, NODE_FEATURE_COLS.index('X Coord [cm]')]
    y_coord = x[:, NODE_FEATURE_COLS.index('Y Coord [cm]')]
    z_coord = x[:, NODE_FEATURE_COLS.index('Z Coord [cm]')]
    pore_radius = np.asarray(x[:, NODE_FEATURE_COLS.index('EqRadius [cm]')], dtype=float)

    unique_edges, unique_edge_attr = _unique_undirected_edges(edge_index, edge_attr)

    # 代表性抽样孔喉，避免几千条边逐条渲染造成卡顿。
    edges_to_plot = unique_edges
    edge_attr_to_plot = unique_edge_attr
    if len(edges_to_plot) > max_edges:
        idx = rng.choice(len(edges_to_plot), max_edges, replace=False)
        edges_to_plot = edges_to_plot[idx]
        if edge_attr_to_plot is not None:
            edge_attr_to_plot = edge_attr_to_plot[idx]

    segments = []
    for s, t in edges_to_plot:
        segments.append([
            [x_coord[s], y_coord[s], z_coord[s]],
            [x_coord[t], y_coord[t], z_coord[t]]
        ])

    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection='3d')

    # 孔喉半径控制线宽，让孔喉关系更清楚。
    if edge_attr_to_plot is not None and len(edge_attr_to_plot) > 0 and "EqRadius [cm]" in EDGE_FEATURE_COLS:
        throat_r = edge_attr_to_plot[:, EDGE_FEATURE_COLS.index("EqRadius [cm]")]
        throat_r = np.nan_to_num(throat_r, nan=0.0, posinf=0.0, neginf=0.0)
        if np.nanmax(throat_r) > np.nanmin(throat_r):
            line_width = 0.35 + 2.20 * (throat_r - np.nanmin(throat_r)) / (np.nanmax(throat_r) - np.nanmin(throat_r) + 1e-12)
        else:
            line_width = np.full(len(edges_to_plot), 0.8)
    else:
        line_width = np.full(len(edges_to_plot), 0.8)

    if segments:
        lc = Line3DCollection(segments, linewidths=line_width, alpha=0.50)
        ax.add_collection3d(lc)

    # 先用淡色小点显示全部孔隙位置，防止抽样球体遗漏整体轮廓。
    ax.scatter(x_coord, y_coord, z_coord, s=4, alpha=0.18)

    # 选择较大的孔隙绘制为真实三维球体，体现三维孔隙而不是二维 marker。
    finite_r = np.nan_to_num(pore_radius, nan=0.0, posinf=0.0, neginf=0.0)
    if len(finite_r) > max_solid_pores:
        # 一半取半径最大的孔隙，一半随机取样，兼顾主孔隙和空间分布。
        n_large = max_solid_pores // 2
        large_idx = np.argsort(finite_r)[-n_large:]
        remaining = np.setdiff1d(np.arange(len(finite_r)), large_idx)
        rand_idx = rng.choice(remaining, max_solid_pores - n_large, replace=False) if len(remaining) >= (max_solid_pores - n_large) else remaining
        pore_idx = np.unique(np.concatenate([large_idx, rand_idx]))
    else:
        pore_idx = np.arange(len(finite_r))

    bbox_diag = np.sqrt(
        (np.nanmax(x_coord) - np.nanmin(x_coord)) ** 2 +
        (np.nanmax(y_coord) - np.nanmin(y_coord)) ** 2 +
        (np.nanmax(z_coord) - np.nanmin(z_coord)) ** 2
    )
    median_r = np.nanmedian(finite_r[finite_r > 0]) if np.any(finite_r > 0) else 1.0
    visual_scale = (bbox_diag * 0.010) / (median_r + 1e-12)
    visual_r = np.clip(finite_r * visual_scale, bbox_diag * 0.0025, bbox_diag * 0.025)

    cmap = cm.get_cmap('viridis')
    norm = Normalize(vmin=np.nanmin(finite_r), vmax=np.nanmax(finite_r) + 1e-12)

    for idx in pore_idx:
        _plot_sphere(
            ax,
            center=(x_coord[idx], y_coord[idx], z_coord[idx]),
            radius=visual_r[idx],
            color_value=finite_r[idx],
            cmap=cmap,
            norm=norm,
            alpha=0.72,
            resolution=sphere_resolution
        )

    _set_axes_equal_3d(ax, x_coord, y_coord, z_coord)

    ax.set_xlabel('X coordinate (cm)')
    ax.set_ylabel('Y coordinate (cm)')
    ax.set_zlabel('Z coordinate (cm)')
    ax.set_title(
        f"3D pore-throat network visualization: {graph['sample_id']}\n"
        f"Pores={len(x_coord)}, throats={len(unique_edges)}, displayed throats={len(edges_to_plot)}, 3D pore spheres={len(pore_idx)}"
    )

    mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
    mappable.set_array(finite_r)
    cbar = plt.colorbar(mappable, ax=ax, shrink=0.65, pad=0.10)
    cbar.set_label('Pore equivalent radius (cm)')

    ax.view_init(elev=24, azim=42)
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        sphere_flag = np.zeros(len(x_coord), dtype=bool)
        sphere_flag[pore_idx] = True
        export_figure_data(
            pd.DataFrame({
                'sample_id': graph['sample_id'],
                'node_id': np.arange(len(x_coord)),
                'x_coord_cm': x_coord,
                'y_coord_cm': y_coord,
                'z_coord_cm': z_coord,
                'pore_equivalent_radius_cm': finite_r,
                'visual_radius': visual_r,
                'is_plotted_as_3d_sphere': sphere_flag
            }),
            save_path,
            suffix='_nodes.csv'
        )
        edge_plot_df = pd.DataFrame({
            'sample_id': graph['sample_id'],
            'src': edges_to_plot[:, 0] if len(edges_to_plot) else np.array([], dtype=int),
            'dst': edges_to_plot[:, 1] if len(edges_to_plot) else np.array([], dtype=int),
            'src_x_coord_cm': [x_coord[s] for s, _ in edges_to_plot],
            'src_y_coord_cm': [y_coord[s] for s, _ in edges_to_plot],
            'src_z_coord_cm': [z_coord[s] for s, _ in edges_to_plot],
            'dst_x_coord_cm': [x_coord[t] for _, t in edges_to_plot],
            'dst_y_coord_cm': [y_coord[t] for _, t in edges_to_plot],
            'dst_z_coord_cm': [z_coord[t] for _, t in edges_to_plot],
            'line_width': line_width
        })
        if edge_attr_to_plot is not None and len(edge_attr_to_plot) > 0:
            for j, name in enumerate(EDGE_FEATURE_COLS[:edge_attr_to_plot.shape[1]]):
                edge_plot_df[name] = edge_attr_to_plot[:, j]
        export_figure_data(edge_plot_df, save_path, suffix='_edges.csv')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()


if len(graphs) > 0:
    visualize_graph_3d(
        graphs[0],
        save_path=RESULT_DIR / f"pore_throat_network_3d_{graphs[0]['sample_id']}.png"
    )


## 6. 特征标准化与标签变换

这里只定义标准化和反标准化函数。标准化器将在训练集划分之后拟合，避免数据泄露。

In [ ]:
# ============================================================
# 6. 特征标准化与标签变换函数
# ============================================================

EPS_K = 1e-12

PERM_TARGET_COLS = ['kx_um2', 'ky_um2', 'kz_um2']
MECH_TARGET_COLS = ['Ex_spring_GPa', 'Ey_spring_GPa', 'Ez_spring_GPa']

PERM_IDXS = [TARGET_COLS.index(c) for c in PERM_TARGET_COLS]
MECH_IDXS = [TARGET_COLS.index(c) for c in MECH_TARGET_COLS]


def transform_perm_y(y_perm):
    """
    渗流标签进入训练前的变换：
    先 log10(k + EPS)，再交给 StandardScaler 标准化。
    """
    y_perm = np.asarray(y_perm, dtype=np.float32).copy()
    return np.log10(np.maximum(y_perm, 0) + EPS_K)


def inverse_perm_y(y_perm_scaled, perm_y_scaler):
    """
    渗流模型输出反归一化：
    标准化空间 -> log10 渗透率空间 -> 真实渗透率。
    """
    y_log = perm_y_scaler.inverse_transform(y_perm_scaled)
    y_real = np.maximum(10 ** y_log - EPS_K, 0)
    return y_real


def inverse_mech_y(y_mech_scaled, mech_y_scaler):
    """
    力学模型输出反归一化：
    标准化空间 -> 真实力学参数。
    """
    return mech_y_scaler.inverse_transform(y_mech_scaled)


print('渗流预测目标:', PERM_TARGET_COLS)
print('力学预测目标:', MECH_TARGET_COLS)
print('说明：标准化器将在训练集划分之后，只用训练集拟合，避免数据泄露。')

## 7. 划分训练集和验证集：岩样级分组划分

本基础版默认采用**岩样级分组划分**：9 个岩样用于训练，2 个岩样用于验证。  
这样可以作为更严格的基础 GNN 对照实验，避免同一岩样的不同切块同时出现在训练集和验证集中。

In [ ]:
# ============================================================
# 7. 数据划分版本1：按岩样分组，9个岩样训练 + 2个岩样验证
# ============================================================

import os
import re
import numpy as np

from collections import Counter

N_VAL_ROCKS = 3

if "SEED" not in globals():
    SEED = 24


# 如果你想固定哪2个岩样作为验证集，把岩样名填到这里；
# 例如：VAL_ROCKS_MANUAL = ["Bandera Brown", "Bentheimer"]
# 如果保持 None，则按 SEED 随机选择2个岩样作为验证集。
VAL_ROCKS_MANUAL = None


def _natural_key_for_split(s):
    s = str(s)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]


def _get_graph_value_for_split(g, keys, default=None):
    for k in keys:
        if isinstance(g, dict) and k in g and g[k] is not None:
            v = g[k]
            if not (isinstance(v, float) and np.isnan(v)):
                return v
    return default


def get_rock_group(g):
    """
    从 graph 字典中识别原始岩样名称。
    注意：本 notebook 第4步生成的变量名是 graphs，不是 graph_data。
    """
    v = _get_graph_value_for_split(
        g,
        ["rock_id", "rock_name", "sample_name", "source_rock", "source_sample"],
        default=None
    )
    if v is not None and str(v).strip() not in ["", "unknown", "nan", "None"]:
        return str(v).strip()

    # 兜底：如果 rock_id 缺失，则尝试从 pnm_xlsx 路径中提取上级目录名。
    p = _get_graph_value_for_split(
        g,
        ["pnm_xlsx", "pnm_excel_path", "excel_path", "file_path", "path", "sample_id"],
        default=None
    )
    if p is not None:
        parts = re.split(r"[\\/]+", str(p))
        if len(parts) >= 2:
            return parts[-2]
        return str(p)

    return "unknown"


if "graphs" not in globals():
    raise NameError("当前 notebook 中没有变量 graphs。请先运行第4步：从 v2 PNM Excel 构建图数据。")

rock_groups = np.array([get_rock_group(g) for g in graphs])
unique_rocks = sorted(np.unique(rock_groups).tolist(), key=_natural_key_for_split)

print("识别到岩样数量:", len(unique_rocks))
print("岩样列表:")
for r in unique_rocks:
    print(" -", r)

if len(unique_rocks) < 3:
    raise ValueError("识别到的岩样数量少于3个，不能进行 9训练/2验证 的岩样级划分。请检查 graphs 中的 rock_id/rock_name 字段。")

if VAL_ROCKS_MANUAL is not None:
    val_rocks = [str(x).strip() for x in VAL_ROCKS_MANUAL]
    missing = [r for r in val_rocks if r not in unique_rocks]
    if missing:
        raise ValueError(f"VAL_ROCKS_MANUAL 中存在未识别岩样: {missing}\n当前可用岩样: {unique_rocks}")
else:
    rng = np.random.default_rng(SEED)
    n_val = min(N_VAL_ROCKS, max(1, len(unique_rocks) - 1))
    val_rocks = sorted(rng.choice(unique_rocks, size=n_val, replace=False).tolist(), key=_natural_key_for_split)

train_rocks = [r for r in unique_rocks if r not in set(val_rocks)]

train_graphs = [g for g, r in zip(graphs, rock_groups) if r in set(train_rocks)]
val_graphs = [g for g, r in zip(graphs, rock_groups) if r in set(val_rocks)]
split_method = f"rock_group_{len(train_rocks)}train_{len(val_rocks)}val"
# Repository-only metadata correction: actual selected groups, not stale 9/2 tag.
SPLIT_MODE = split_method

# 写入数据集划分标签，供后续 GNN 图级嵌入可视化使用。
for g in train_graphs:
    g["split"] = "train"
for g in val_graphs:
    g["split"] = "val"

print("\n数据划分方式:", split_method)
print("训练集数量:", len(train_graphs))
print("验证集数量:", len(val_graphs))

print("\n训练集岩样数:", len(train_rocks))
for r in train_rocks:
    print(" -", r)

print("\n验证集岩样数:", len(val_rocks))
for r in val_rocks:
    print(" -", r)

print("\n各岩样图数量:")
for r, n in sorted(Counter(rock_groups).items(), key=lambda x: _natural_key_for_split(x[0])):
    tag = "验证集" if r in set(val_rocks) else "训练集"
    print(f"{r}: {n} ({tag})")

if len(train_graphs) == 0 or len(val_graphs) == 0:
    raise ValueError("训练集或验证集为空，请检查样本数量和岩样分组字段。")

In [ ]:
# ============================================================
# 7.1 仅使用训练集拟合标准化器，并分别构建渗流/力学输入
# ============================================================
# 改进原则：
# 1. 渗流模型使用：基础 global_descriptors + 渗流路径描述符；
# 2. 力学模型仍使用原有 global_descriptors，不引入渗流专用路径特征；
# 3. 所有标准化器只在训练集拟合，避免数据泄漏。

all_train_node_features = np.vstack(
    [g["x"] for g in train_graphs]
)
all_train_edge_features = np.vstack(
    [g["edge_attr"] for g in train_graphs]
)
all_train_global_features = np.vstack(
    [g["global_features"] for g in train_graphs]
)

node_scaler = StandardScaler()
edge_scaler = StandardScaler()

node_scaler.fit(all_train_node_features)
edge_scaler.fit(all_train_edge_features)

perm_global_feature_cols = list(GLOBAL_FEATURE_COLS)
mech_global_feature_cols = list(BASE_GLOBAL_FEATURE_COLS)

perm_global_indices = [
    GLOBAL_FEATURE_COLS.index(c)
    for c in perm_global_feature_cols
]
mech_global_indices = [
    GLOBAL_FEATURE_COLS.index(c)
    for c in mech_global_feature_cols
]

perm_global_feature_scaler = StandardScaler()
mech_global_feature_scaler = StandardScaler()

perm_global_feature_scaler.fit(
    all_train_global_features[:, perm_global_indices]
)
mech_global_feature_scaler.fit(
    all_train_global_features[:, mech_global_indices]
)

# 渗流和力学标签分别标准化。
Y_train_raw = np.vstack(
    [g["y"] for g in train_graphs]
).astype(np.float32)

Y_train_perm = Y_train_raw[:, PERM_IDXS]
Y_train_perm_trans = transform_perm_y(Y_train_perm)
Y_train_mech = Y_train_raw[:, MECH_IDXS]

perm_y_scaler = StandardScaler()
mech_y_scaler = StandardScaler()

perm_y_scaler.fit(Y_train_perm_trans)
mech_y_scaler.fit(Y_train_mech)


def apply_scalers_to_graphs(graph_list):
    for g in graph_list:
        node_scaled = node_scaler.transform(
            g["x"]
        ).astype(np.float32)
        edge_scaled = edge_scaler.transform(
            g["edge_attr"]
        ).astype(np.float32)

        all_global = np.asarray(
            g["global_features"],
            dtype=np.float32
        ).reshape(1, -1)

        perm_global_scaled = (
            perm_global_feature_scaler
            .transform(all_global[:, perm_global_indices])
            .astype(np.float32)
            .reshape(-1)
        )
        mech_global_scaled = (
            mech_global_feature_scaler
            .transform(all_global[:, mech_global_indices])
            .astype(np.float32)
            .reshape(-1)
        )

        # 渗流输入：加入路径级 flow descriptors。
        perm_global_repeated = np.repeat(
            perm_global_scaled.reshape(1, -1),
            node_scaled.shape[0],
            axis=0
        )
        g["x_perm_scaled"] = np.concatenate(
            [node_scaled, perm_global_repeated],
            axis=1
        ).astype(np.float32)

        # 力学输入：保持原 global_descriptors，不加入 flow descriptors。
        mech_global_repeated = np.repeat(
            mech_global_scaled.reshape(1, -1),
            node_scaled.shape[0],
            axis=0
        )
        g["x_mech_scaled"] = np.concatenate(
            [node_scaled, mech_global_repeated],
            axis=1
        ).astype(np.float32)

        # 兼容后续按节点数组批的代码。
        g["x_scaled"] = g["x_perm_scaled"]

        g["edge_attr_scaled"] = edge_scaled
        g["perm_global_features_scaled"] = perm_global_scaled
        g["mech_global_features_scaled"] = mech_global_scaled
        g["global_features_scaled"] = perm_global_scaled

        y_all = g["y"].copy().reshape(1, -1)

        y_perm = y_all[:, PERM_IDXS]
        y_perm_trans = transform_perm_y(y_perm)
        g["y_perm_scaled"] = (
            perm_y_scaler
            .transform(y_perm_trans)
            .astype(np.float32)
            .reshape(-1)
        )

        y_mech = y_all[:, MECH_IDXS]
        g["y_mech_scaled"] = (
            mech_y_scaler
            .transform(y_mech)
            .astype(np.float32)
            .reshape(-1)
        )

    return graph_list


train_graphs = apply_scalers_to_graphs(train_graphs)
val_graphs = apply_scalers_to_graphs(val_graphs)
graphs = apply_scalers_to_graphs(graphs)

print("节点原始特征维度:", train_graphs[0]["x"].shape[1])
print(
    "基础 global_descriptors 数量:",
    len(BASE_GLOBAL_FEATURE_COLS)
)
print(
    "渗流专用路径描述符数量:",
    len(FLOW_GLOBAL_FEATURE_COLS)
)
print(
    "渗流模型最终节点输入维度:",
    train_graphs[0]["x_perm_scaled"].shape[1]
)
print(
    "力学模型最终节点输入维度:",
    train_graphs[0]["x_mech_scaled"].shape[1]
)
print(
    "边特征维度:",
    train_graphs[0]["edge_attr_scaled"].shape[1]
)
print("渗流模型输出维度:", len(PERM_TARGET_COLS))
print("力学模型输出维度:", len(MECH_TARGET_COLS))
print(
    "标准化完成：flow descriptors 仅进入渗流模型，"
    "力学模型保持原输入体系。"
)



## 8. 构建 TensorFlow 图数据生成器

因为每个孔隙网络的节点数和边数不同，这里采用自定义训练循环。每次输入一个图，适合先跑通流程。后续如果数据量很大，可以再改成 mini-batch padding 或 GraphTensor。


In [ ]:
# ============================================================
# 8. TensorFlow 图数据准备函数
# ============================================================

def graph_to_tensors(g, task="perm"):
    """
    task="perm"：
        使用 x_perm_scaled，包含基础 global_descriptors
        和渗流专用路径描述符；
    task="mech"：
        使用 x_mech_scaled，仅包含原有 global_descriptors。
    """
    if task == "perm":
        x_np = g["x_perm_scaled"]
        y_np = g["y_perm_scaled"]
    elif task == "mech":
        x_np = g["x_mech_scaled"]
        y_np = g["y_mech_scaled"]
    else:
        raise ValueError("task 只能是 'perm' 或 'mech'")

    x = tf.convert_to_tensor(
        x_np, dtype=tf.float32
    )
    edge_index = tf.convert_to_tensor(
        g["edge_index"], dtype=tf.int32
    )

    if "edge_attr_scaled" in g:
        edge_attr_np = g["edge_attr_scaled"]
    elif "edge_attr" in g:
        edge_attr_np = g["edge_attr"]
    else:
        edge_attr_np = np.zeros(
            (np.asarray(g["edge_index"]).shape[0], 1),
            dtype=np.float32
        )

    edge_attr = tf.convert_to_tensor(
        edge_attr_np, dtype=tf.float32
    )
    y = tf.convert_to_tensor(
        y_np, dtype=tf.float32
    )

    return x, edge_index, edge_attr, y


def add_self_loops(edge_index, num_nodes):
    self_edges = tf.stack(
        [tf.range(num_nodes), tf.range(num_nodes)],
        axis=1
    )
    self_edges = tf.cast(self_edges, tf.int32)
    return tf.concat(
        [edge_index, self_edges],
        axis=0
    )


def normalize_messages(messages, dst, num_nodes):
    out = tf.math.unsorted_segment_sum(
        messages, dst, num_nodes
    )
    deg = tf.math.unsorted_segment_sum(
        tf.ones(
            (tf.shape(dst)[0], 1),
            dtype=tf.float32
        ),
        dst,
        num_nodes
    )
    out = out / tf.maximum(deg, 1.0)
    return out


## 9. 定义 PM-GNN：global_descriptors + flow path descriptors


In [ ]:
# ============================================================
# 9. 定义 PM-GNN：global_descriptors + flow path descriptors
# ============================================================
# 设计原则：
# 1. 保留原始 BasicGraphConv、hidden_dim、num_layers、dropout 和 mean pooling；
# 2. 渗流模型加入 flow-specific path descriptors；
# 3. 力学模型仍使用原有 global_descriptors；
# 4. 不使用 enhanced_MP、attention readout 或 edge_attr 消息传递；
# 5. 渗流模型仍同时输出 kx、ky、kz，便于只检验 flow descriptors 的贡献。

PM_HIDDEN_DIM = 64
PM_NUM_LAYERS = 2
PM_DROPOUT = 0.1

USE_GLOBAL_DESCRIPTORS = True
USE_FLOW_PATH_DESCRIPTORS = True
USE_DIRECTION_CONDITIONED_PERM_HEAD = False
USE_ENHANCED_MP = False


class BasicGraphConv(layers.Layer):
    """
    与原始 GNN 一致的基础图卷积：
    邻居均值聚合 + Dense 更新。
    """
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.dense = layers.Dense(
            hidden_dim,
            activation="relu"
        )
        self.dropout = layers.Dropout(dropout)

    def call(
        self,
        h,
        edge_index,
        edge_attr=None,
        training=False
    ):
        num_nodes = tf.shape(h)[0]

        edge_index2 = add_self_loops(
            edge_index, num_nodes
        )
        src = edge_index2[:, 0]
        dst = edge_index2[:, 1]

        h_src = tf.gather(h, src)
        agg = normalize_messages(
            h_src, dst, num_nodes
        )

        h_new = self.dense(
            agg, training=training
        )
        h_new = self.dropout(
            h_new, training=training
        )
        return h_new


class PMGNN(keras.Model):
    """
    PM-GNN：
    [节点基础特征 || global_descriptors]
      -> BasicGraphConv
      -> mean pooling
      -> 回归头。

    当 direction_conditioned=True 时：
    graph embedding 与 x/y/z one-hot 分别组合，
    共享一个标量回归头生成 kx、ky、kz。
    """
    def __init__(
        self,
        node_dim,
        hidden_dim=PM_HIDDEN_DIM,
        num_layers=PM_NUM_LAYERS,
        out_dim=3,
        dropout=PM_DROPOUT,
        direction_conditioned=False
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.out_dim = out_dim
        self.direction_conditioned = direction_conditioned

        self.use_global_descriptors = (
            USE_GLOBAL_DESCRIPTORS
        )
        self.use_flow_path_descriptors = (
            USE_FLOW_PATH_DESCRIPTORS
        )
        self.use_enhanced_mp = USE_ENHANCED_MP

        self.encoder = keras.Sequential([
            layers.Dense(
                hidden_dim,
                activation="relu"
            ),
            layers.Dropout(dropout)
        ])

        self.convs = [
            BasicGraphConv(
                hidden_dim,
                dropout=dropout
            )
            for _ in range(num_layers)
        ]

        if self.direction_conditioned:
            if out_dim != 3:
                raise ValueError(
                    "方向条件化渗流头要求 out_dim=3。"
                )

            self.direction_encoder = keras.Sequential([
                layers.Dense(
                    16,
                    activation="relu"
                )
            ])
            self.scalar_readout = keras.Sequential([
                layers.Dense(
                    hidden_dim,
                    activation="relu"
                ),
                layers.Dropout(dropout),
                layers.Dense(1)
            ])
            self.readout = None
        else:
            self.direction_encoder = None
            self.scalar_readout = None
            self.readout = keras.Sequential([
                layers.Dense(
                    hidden_dim,
                    activation="relu"
                ),
                layers.Dropout(dropout),
                layers.Dense(out_dim)
            ])

    def call(
        self,
        x,
        edge_index,
        edge_attr=None,
        batch_index=None,
        training=False,
        return_embedding=False
    ):
        h = self.encoder(
            x, training=training
        )

        for conv in self.convs:
            h = conv(
                h,
                edge_index,
                edge_attr=edge_attr,
                training=training
            )

        if batch_index is not None:
            graph_emb = tf.math.segment_mean(
                h, batch_index
            )
        else:
            graph_emb = tf.reduce_mean(
                h,
                axis=0,
                keepdims=True
            )

        if self.direction_conditioned:
            batch_size = tf.shape(graph_emb)[0]

            direction_one_hot = tf.eye(
                3,
                dtype=graph_emb.dtype
            )
            direction_one_hot = tf.tile(
                direction_one_hot,
                [batch_size, 1]
            )

            graph_repeated = tf.repeat(
                graph_emb,
                repeats=3,
                axis=0
            )

            direction_emb = self.direction_encoder(
                direction_one_hot,
                training=training
            )

            fused = tf.concat(
                [graph_repeated, direction_emb],
                axis=1
            )
            scalar_pred = self.scalar_readout(
                fused,
                training=training
            )
            y_pred = tf.reshape(
                scalar_pred,
                [batch_size, 3]
            )
        else:
            y_pred = self.readout(
                graph_emb,
                training=training
            )

        if return_embedding:
            return y_pred, graph_emb
        return y_pred


PoreTopologyGNN = PMGNN


def print_pm_gnn_model_info(
    model,
    model_name,
    node_dim,
    out_dim
):
    print(f"\n{model_name}：PM-GNN 模型结构")
    print("-" * 76)
    print(f"输入节点特征维度: {node_dim}")
    print(f"隐藏维度: {model.hidden_dim}")
    print(f"图卷积层数: {model.num_layers}")
    print("图读出方式: mean pooling")
    print(f"输出维度: {out_dim}")
    print(
        "使用 global_descriptors:",
        model.use_global_descriptors
    )
    print(
        "使用 flow path descriptors:",
        model.use_flow_path_descriptors
    )
    print(
        "方向条件化渗流头:",
        model.direction_conditioned
    )
    print(
        "使用 enhanced_MP:",
        model.use_enhanced_mp
    )
    print("-" * 76)


perm_node_dim = (
    train_graphs[0]["x_perm_scaled"].shape[1]
)
mech_node_dim = (
    train_graphs[0]["x_mech_scaled"].shape[1]
)

perm_model = PMGNN(
    node_dim=perm_node_dim,
    hidden_dim=PM_HIDDEN_DIM,
    num_layers=PM_NUM_LAYERS,
    out_dim=len(PERM_TARGET_COLS),
    dropout=PM_DROPOUT,
    direction_conditioned=(
        USE_DIRECTION_CONDITIONED_PERM_HEAD
    )
)

mech_model = PMGNN(
    node_dim=mech_node_dim,
    hidden_dim=PM_HIDDEN_DIM,
    num_layers=PM_NUM_LAYERS,
    out_dim=len(MECH_TARGET_COLS),
    dropout=PM_DROPOUT,
    direction_conditioned=False
)

perm_optimizer = keras.optimizers.Adam(
    learning_rate=1e-3
)
mech_optimizer = keras.optimizers.Adam(
    learning_rate=1e-3
)
loss_fn = keras.losses.MeanSquaredError()

x0, ei0, ea0, y0 = graph_to_tensors(
    train_graphs[0],
    task="perm"
)
_ = perm_model(
    x0,
    ei0,
    ea0,
    training=False
)

x0, ei0, ea0, y0 = graph_to_tensors(
    train_graphs[0],
    task="mech"
)
_ = mech_model(
    x0,
    ei0,
    ea0,
    training=False
)

print_pm_gnn_model_info(
    perm_model,
    "渗流模型",
    perm_node_dim,
    len(PERM_TARGET_COLS)
)
perm_model.summary()

print_pm_gnn_model_info(
    mech_model,
    "力学模型",
    mech_node_dim,
    len(MECH_TARGET_COLS)
)
mech_model.summary()


## 10. 训练改进后的 PM-GNN 模型


In [ ]:
# ============================================================
# 新增：图批处理工具函数（核心加速）
# ============================================================
def batch_graphs(graph_list, task):
    """
    将多个图拼接成一个批处理大图
    返回: x, edge_index, edge_attr, y, batch_index
    """
    x_list = []
    edge_index_list = []
    edge_attr_list = []
    y_list = []
    batch_index_list = []
    
    node_offset = 0
    for i, g in enumerate(graph_list):
        x, edge_index, edge_attr, y = graph_to_tensors(g, task=task)
        
        # 边索引偏移（关键！避免不同图的节点ID冲突）
        edge_index = edge_index + node_offset
        
        x_list.append(x)
        edge_index_list.append(edge_index)
        edge_attr_list.append(edge_attr)
        y_list.append(y)
        batch_index_list.append(tf.fill((x.shape[0],), i))
        
        node_offset += x.shape[0]
    
    # 拼接所有张量
    x = tf.concat(x_list, axis=0)
    edge_index = tf.concat(edge_index_list, axis=0)
    edge_attr = tf.concat(edge_attr_list, axis=0)
    y = tf.stack(y_list, axis=0)
    batch_index = tf.concat(batch_index_list, axis=0)
    
    return x, edge_index, edge_attr, y, batch_index

In [ ]:
# ============================================================
# 10. PM-GNN训练循环：Early Stopping + 自适应学习率 + 动态批处理
# ============================================================
import time
from tqdm import tqdm
import random
import numpy as np
import tensorflow as tf
import gc


def train_one_batch(current_model, current_optimizer, x, edge_index, edge_attr, y, batch_index):
    with tf.GradientTape() as tape:
        pred = current_model(x, edge_index, edge_attr, batch_index, training=True)
        loss = loss_fn(y, pred)

        # 兼容未来如果在层里加入 regularizer 的情况。
        if current_model.losses:
            loss = loss + tf.add_n(current_model.losses)

    grads = tape.gradient(loss, current_model.trainable_variables)
    grads = [tf.clip_by_norm(g, 5.0) if g is not None else None for g in grads]
    current_optimizer.apply_gradients(zip(grads, current_model.trainable_variables))
    return loss


def make_memory_safe_batches(graph_list, max_graphs=32, max_nodes=2500, max_edges=20000):
    """
    按“图数量 + 节点数 + 边数”动态组批，避免某个 batch 中图太多导致 GPU OOM。

    说明：
    原 notebook 固定 BATCH_SIZE=64。虽然单个图不大，但 64 个图拼成一个大图后，
    消息传递层需要同时保存节点、边、梯度和中间激活，低显存 GPU 容易在第一轮 OOM。
    """
    batches = []
    current = []
    n_nodes = 0
    n_edges = 0

    for g in graph_list:
        g_nodes = int(g["x_scaled"].shape[0])
        g_edges = int(g["edge_index"].shape[0])

        exceed_graphs = len(current) >= max_graphs
        exceed_nodes = current and (n_nodes + g_nodes > max_nodes)
        exceed_edges = current and (n_edges + g_edges > max_edges)

        if exceed_graphs or exceed_nodes or exceed_edges:
            batches.append(current)
            current = []
            n_nodes = 0
            n_edges = 0

        current.append(g)
        n_nodes += g_nodes
        n_edges += g_edges

    if current:
        batches.append(current)

    return batches


def evaluate_graphs(current_model, graph_list, task, batch_size=32, max_nodes=2500, max_edges=20000):
    losses = []
    eval_batches = make_memory_safe_batches(
        graph_list,
        max_graphs=batch_size,
        max_nodes=max_nodes,
        max_edges=max_edges
    )
    for batch_graphs_list in eval_batches:
        x, edge_index, edge_attr, y, batch_index = batch_graphs(batch_graphs_list, task=task)
        pred = current_model(x, edge_index, edge_attr, batch_index, training=False)
        loss = loss_fn(y, pred).numpy()
        losses.append(loss * len(batch_graphs_list))

    return float(np.sum(losses) / len(graph_list)) if losses else np.nan


def _get_optimizer_lr(optimizer):
    try:
        return float(tf.keras.backend.get_value(optimizer.learning_rate))
    except Exception:
        return float(tf.keras.backend.get_value(optimizer.lr))


def _set_optimizer_lr(optimizer, lr_value):
    try:
        tf.keras.backend.set_value(optimizer.learning_rate, lr_value)
    except Exception:
        tf.keras.backend.set_value(optimizer.lr, lr_value)


def train_task_model(
    task_name,
    current_model,
    current_optimizer,
    task,
    epochs=300,
    batch_size=16,
    log_interval=1,
    patience=30,
    min_delta=1e-4,
    reduce_lr_patience=10,
    reduce_lr_factor=0.5,
    min_lr=1e-5,
    max_nodes_per_batch=800,
    max_edges_per_batch=5000
):
    """
    显存友好版训练函数。

    新增/保留：
    1. Early Stopping：验证集损失连续 patience 轮没有有效下降则停止；
    2. Restore Best Weights：训练结束后自动恢复验证集最优权重；
    3. ReduceLROnPlateau：验证集停滞时降低学习率；
    4. 梯度裁剪：减少小数据图回归中的不稳定更新；
    5. 动态小批量：同时限制 batch 内图数量、节点数和边数，避免 GPU OOM。
    """
    train_losses = []
    val_losses = []
    lr_history = []

    best_val = np.inf
    best_epoch = 0
    best_weights = None
    wait = 0
    lr_wait = 0

    total_start_time = time.time()

    print(f'\n{"="*50}')
    print(f'开始训练：{task_name}')
    print(f'训练集样本数：{len(train_graphs)} | 验证集样本数：{len(val_graphs)}')
    print(
        f'最大轮次：{epochs} | 单批最多图数：{batch_size} | '
        f'单批最多节点数：{max_nodes_per_batch} | 单批最多边数：{max_edges_per_batch} | '
        f'EarlyStopping patience={patience}'
    )
    print(f'{"="*50}')

    for epoch in range(1, epochs + 1):
        epoch_train_graphs = train_graphs.copy()
        random.shuffle(epoch_train_graphs)

        batches = make_memory_safe_batches(
            epoch_train_graphs,
            max_graphs=batch_size,
            max_nodes=max_nodes_per_batch,
            max_edges=max_edges_per_batch
        )
        batch_losses = []

        pbar = tqdm(
            batches,
            desc=f'{task_name} 轮次 {epoch:03d}/{epochs}',
            bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, 损失={postfix}]',
            postfix='0.000000'
        )

        try:
            for batch in pbar:
                x, edge_index, edge_attr, y, batch_index = batch_graphs(batch, task=task)
                loss = train_one_batch(current_model, current_optimizer, x, edge_index, edge_attr, y, batch_index)
                loss_val = float(loss.numpy())
                batch_losses.append(loss_val)
                pbar.set_postfix_str(f'{np.mean(batch_losses):.6f}')

        except tf.errors.ResourceExhaustedError as e:
            pbar.close()
            gc.collect()
            msg = (
                "\n检测到 GPU 显存不足（ResourceExhaustedError）。\n"
                "请重启内核后重新运行本 notebook；如果仍然报错，把第0节中的 FORCE_CPU_TRAINING 改成 True。\n"
                "也可以继续减小第10节中的 BATCH_SIZE、MAX_NODES_PER_BATCH 和 MAX_EDGES_PER_BATCH。"
            )
            raise RuntimeError(msg) from e

        pbar.close()

        train_loss = float(np.mean(batch_losses))
        val_loss = evaluate_graphs(
            current_model,
            val_graphs,
            task=task,
            batch_size=batch_size,
            max_nodes=max_nodes_per_batch,
            max_edges=max_edges_per_batch
        )
        current_lr = _get_optimizer_lr(current_optimizer)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        lr_history.append(current_lr)

        improved = val_loss < (best_val - min_delta)

        if improved:
            best_val = val_loss
            best_epoch = epoch
            best_weights = current_model.get_weights()
            wait = 0
            lr_wait = 0
        else:
            wait += 1
            lr_wait += 1

        if lr_wait >= reduce_lr_patience:
            old_lr = _get_optimizer_lr(current_optimizer)
            new_lr = max(old_lr * reduce_lr_factor, min_lr)
            if new_lr < old_lr:
                _set_optimizer_lr(current_optimizer, new_lr)
                print(f'学习率降低：{old_lr:.2e} -> {new_lr:.2e}')
            lr_wait = 0

        if epoch % log_interval == 0:
            best_info = f' | 当前最优验证损失: {best_val:.6f} (第{best_epoch}轮)'
            print(
                f'✅ 轮次 {epoch:03d} 完成 | 训练损失: {train_loss:.6f} | '
                f'验证损失: {val_loss:.6f} | LR: {current_lr:.2e}{best_info}'
            )

        if wait >= patience:
            print(f'⏹ Early Stopping：验证损失连续 {patience} 轮未提升，停止于第 {epoch} 轮。')
            break

    if best_weights is not None:
        current_model.set_weights(best_weights)
        print(f'已恢复 {task_name} 第 {best_epoch} 轮的最优权重，最佳验证损失: {best_val:.6f}')

    total_time = time.time() - total_start_time
    print(f'\n{"="*50}')
    print(f'{task_name} 训练完成！总耗时: {total_time:.2f} 秒')
    print(f'实际训练轮数: {len(train_losses)}')
    print(f'最佳轮次: {best_epoch} | 最佳验证损失: {best_val:.6f}')
    print(f'最终训练损失: {train_losses[-1]:.6f} | 最终验证损失: {val_losses[-1]:.6f}')
    print(f'{"="*50}\n')

    history_df = pd.DataFrame({
        "epoch": np.arange(1, len(train_losses) + 1),
        "train_loss": train_losses,
        "val_loss": val_losses,
        "learning_rate": lr_history,
        "task": task_name,
        "best_epoch": best_epoch,
        "best_val_loss": best_val
    })
    history_df.to_csv(RESULT_DIR / f"training_history_{task}.csv", index=False, encoding="utf-8-sig")

    return train_losses, val_losses


# 训练配置：保留基础 GNN 的训练显示和早停逻辑
# 如果仍然 OOM，优先把 BATCH_SIZE 改为 16；其次把 MAX_EDGES_PER_BATCH 改为 10000。
EPOCHS = 300
BATCH_SIZE = 64
MAX_NODES_PER_BATCH = 80000
MAX_EDGES_PER_BATCH = 200000
LOG_INTERVAL = 1
EARLY_STOP_PATIENCE = 20
MIN_DELTA = 1e-4

# 训练渗流模型
perm_train_losses, perm_val_losses = train_task_model(
    task_name='渗流模型',
    current_model=perm_model,
    current_optimizer=perm_optimizer,
    task='perm',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    log_interval=LOG_INTERVAL,
    patience=EARLY_STOP_PATIENCE,
    min_delta=MIN_DELTA,
    reduce_lr_patience=10,
    reduce_lr_factor=0.5,
    min_lr=1e-5,
    max_nodes_per_batch=MAX_NODES_PER_BATCH,
    max_edges_per_batch=MAX_EDGES_PER_BATCH
)

# 训练力学模型
mech_train_losses, mech_val_losses = train_task_model(
    task_name='力学模型',
    current_model=mech_model,
    current_optimizer=mech_optimizer,
    task='mech',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    log_interval=LOG_INTERVAL,
    patience=EARLY_STOP_PATIENCE,
    min_delta=MIN_DELTA,
    reduce_lr_patience=10,
    reduce_lr_factor=0.5,
    min_lr=1e-5,
    max_nodes_per_batch=MAX_NODES_PER_BATCH,
    max_edges_per_batch=MAX_EDGES_PER_BATCH
)


In [ ]:
# ============================================================
# 10.1 Training loss curves（修复版：分开绘制+自动处理不同轮次）
# ============================================================

# 1. 绘制渗流模型损失曲线
plt.figure(figsize=(8, 5))
plt.plot(perm_train_losses, label='Training loss', linewidth=2, color='#1f77b4')
plt.plot(perm_val_losses, label='Validation loss', linewidth=2, color='#ff7f0e')
plt.xlabel('Epoch')
plt.ylabel('MSE loss (standardized space)')
plt.title('Training loss curve - Permeability model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# 同时输出PNG和SVG
perm_loss_path = RESULT_DIR / 'training_loss_curve_permeability.png'
plt.savefig(perm_loss_path, dpi=300, bbox_inches='tight')
plt.savefig(perm_loss_path.with_suffix('.svg'), format='svg', bbox_inches='tight')
plt.show()

# 导出渗流模型损失数据
perm_loss_df = pd.DataFrame({
    "epoch": np.arange(1, len(perm_train_losses) + 1),
    "training_loss": perm_train_losses,
    "validation_loss": perm_val_losses
})
export_figure_data(perm_loss_df, perm_loss_path)

# 2. 绘制力学模型损失曲线
plt.figure(figsize=(8, 5))
plt.plot(mech_train_losses, label='Training loss', linewidth=2, color='#2ca02c')
plt.plot(mech_val_losses, label='Validation loss', linewidth=2, color='#d62728')
plt.xlabel('Epoch')
plt.ylabel('MSE loss (standardized space)')
plt.title('Training loss curve - Mechanics model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# 同时输出PNG和SVG
mech_loss_path = RESULT_DIR / 'training_loss_curve_mechanics.png'
plt.savefig(mech_loss_path, dpi=300, bbox_inches='tight')
plt.savefig(mech_loss_path.with_suffix('.svg'), format='svg', bbox_inches='tight')
plt.show()

# 导出力学模型损失数据
mech_loss_df = pd.DataFrame({
    "epoch": np.arange(1, len(mech_train_losses) + 1),
    "training_loss": mech_train_losses,
    "validation_loss": mech_val_losses
})
export_figure_data(mech_loss_df, mech_loss_path)

# 打印训练轮次信息
print(f"\n✅ 损失曲线绘制完成！")
print(f"渗流模型实际训练轮次: {len(perm_train_losses)}")
print(f"力学模型实际训练轮次: {len(mech_train_losses)}")
print(f"文件已保存到: {RESULT_DIR}")


## 11. 测试集预测与反标准化

这里会把模型输出还原到真实单位。注意：前三个渗透率先从标准化空间还原到 `log10(k)`，再反变换为真实渗透率。


In [ ]:
# ============================================================
# 11. PM-GNN 验证集预测与反归一化
# ============================================================

def predict_graphs_scaled(model, graph_list, task='perm', batch_size=32):
    preds_scaled = []
    y_scaled = []
    sample_ids = []
    rock_ids = []

    for i in range(0, len(graph_list), batch_size):
        batch = graph_list[i:i+batch_size]
        x, edge_index, edge_attr, y, batch_index = batch_graphs(batch, task=task)
        pred = model(x, edge_index, edge_attr, batch_index, training=False).numpy()
        preds_scaled.append(pred)
        y_scaled.append(y.numpy())

        for g in batch:
            sample_ids.append(g['sample_id'])
            rock_ids.append(g.get('rock_id', 'unknown'))

    return sample_ids, rock_ids, np.vstack(y_scaled), np.vstack(preds_scaled)


def predict_perm_graphs(graph_list):
    sample_ids, rock_ids, y_scaled, preds_scaled = predict_graphs_scaled(
        perm_model, graph_list, task='perm', batch_size=32
    )
    preds_real = inverse_perm_y(preds_scaled, perm_y_scaler)
    y_real = inverse_perm_y(y_scaled, perm_y_scaler)
    return sample_ids, rock_ids, y_real, preds_real


def predict_mech_graphs(graph_list):
    sample_ids, rock_ids, y_scaled, preds_scaled = predict_graphs_scaled(
        mech_model, graph_list, task='mech', batch_size=32
    )
    preds_real = inverse_mech_y(preds_scaled, mech_y_scaler)
    y_real = inverse_mech_y(y_scaled, mech_y_scaler)
    return sample_ids, rock_ids, y_real, preds_real


sample_ids_perm, rock_ids_perm, y_true_perm, y_pred_perm = predict_perm_graphs(val_graphs)
sample_ids_mech, rock_ids_mech, y_true_mech, y_pred_mech = predict_mech_graphs(val_graphs)

sample_ids = sample_ids_perm
y_true = np.hstack([y_true_perm, y_true_mech])
y_pred = np.hstack([y_pred_perm, y_pred_mech])

result_df = pd.DataFrame({'sample_id': sample_ids, 'rock_id': rock_ids_perm, 'split_mode': SPLIT_MODE})

for i, col in enumerate(PERM_TARGET_COLS):
    result_df[f'{col}_真实值'] = y_true_perm[:, i]
    result_df[f'{col}_预测值'] = y_pred_perm[:, i]
    result_df[f'{col}_残差'] = y_pred_perm[:, i] - y_true_perm[:, i]

for i, col in enumerate(MECH_TARGET_COLS):
    result_df[f'{col}_真实值'] = y_true_mech[:, i]
    result_df[f'{col}_预测值'] = y_pred_mech[:, i]
    result_df[f'{col}_残差'] = y_pred_mech[:, i] - y_true_mech[:, i]

result_path = RESULT_DIR / f'validation_prediction_results_tensorflow_pm_gnn_flow_path_descriptors_{SPLIT_MODE}.csv'
result_df.to_csv(result_path, index=False, encoding='utf-8-sig')

print('预测结果已保存:', result_path)
print('注意：这里保存的是反归一化后的真实物理量，不是标准化值。')
display(result_df.head())



## 12. 评价指标与中文可视化

输出每个目标的 `R²`、`MAE` 和 `RMSE`，并绘制真实值—预测值散点图。


In [ ]:
# ============================================================
# 12. PM-GNN 完整回归评价指标（含学术论文标准指标）
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    explained_variance_score, median_absolute_error, max_error
)


def safe_pearsonr(y_true, y_pred):
    if len(y_true) <= 2:
        return np.nan, np.nan
    if np.nanstd(y_true) == 0 or np.nanstd(y_pred) == 0:
        return np.nan, np.nan
    try:
        return pearsonr(y_true, y_pred)
    except Exception:
        return np.nan, np.nan


def safe_round(x, ndigits):
    try:
        if pd.isna(x):
            return np.nan
        return round(float(x), ndigits)
    except Exception:
        return np.nan


def calculate_regression_metrics(y_true, y_pred, col_name, n_features=1):
    """计算完整的回归评价指标。"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    n_samples = len(y_true)
    if n_samples == 0:
        return {"预测目标": col_name, "样本数": 0}

    r2 = r2_score(y_true, y_pred) if n_samples > 1 else np.nan
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    r, p_value = safe_pearsonr(y_true, y_pred)
    explained_var = explained_variance_score(y_true, y_pred) if n_samples > 1 else np.nan
    medae = median_absolute_error(y_true, y_pred)
    maxae = max_error(y_true, y_pred)

    if n_samples > n_features + 1 and np.isfinite(r2):
        adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    else:
        adj_r2 = np.nan

    epsilon = 1e-8
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    smape = np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + epsilon)) * 100
    rrmse = (rmse / (np.mean(y_true) + epsilon)) * 100
    bias = np.mean(y_pred) - np.mean(y_true)

    return {
        "预测目标": col_name,
        "样本数": n_samples,
        "R²": safe_round(r2, 4),
        "调整R²": safe_round(adj_r2, 4),
        "皮尔逊r": safe_round(r, 4),
        "p值": p_value,
        "解释方差": safe_round(explained_var, 4),
        "MAE": safe_round(mae, 6),
        "RMSE": safe_round(rmse, 6),
        "MedAE": safe_round(medae, 6),
        "MaxAE": safe_round(maxae, 6),
        "MAPE(%)": safe_round(mape, 2),
        "SMAPE(%)": safe_round(smape, 2),
        "RRMSE(%)": safe_round(rrmse, 2),
        "预测偏差": safe_round(bias, 6)
    }


metrics = []
for i, col in enumerate(TARGET_COLS):
    true_i = y_true[:, i]
    pred_i = y_pred[:, i]
    metrics.append(calculate_regression_metrics(true_i, pred_i, col, n_features=10))

numeric_metric_cols = ["R²", "调整R²", "皮尔逊r", "解释方差", "MAE", "RMSE", "MedAE", "MaxAE", "MAPE(%)", "SMAPE(%)", "RRMSE(%)", "预测偏差"]
avg_metrics = {"预测目标": "平均", "样本数": np.mean([m.get("样本数", np.nan) for m in metrics]), "p值": "-"}
for c in numeric_metric_cols:
    avg_metrics[c] = safe_round(np.nanmean([m.get(c, np.nan) for m in metrics]), 6 if c in ["MAE", "RMSE", "MedAE", "MaxAE", "预测偏差"] else 4)
metrics.append(avg_metrics)

metrics_df = pd.DataFrame(metrics)
metrics_df["p值"] = metrics_df["p值"].apply(lambda x: f"{x:.2e}" if isinstance(x, (float, np.floating)) and np.isfinite(x) else x)

metrics_path = RESULT_DIR / f"metrics_gnn_complete_academic_pm_gnn_flow_path_descriptors_{SPLIT_MODE}.csv"
metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
print("完整学术评价指标已保存:", metrics_path)
display(metrics_df)


In [ ]:
# ============================================================
# 12.1 True vs. predicted scatter plots
# ============================================================

TARGET_DISPLAY_NAMES = {
    'kx_um2': 'X-direction permeability',
    'ky_um2': 'Y-direction permeability',
    'kz_um2': 'Z-direction permeability',
    'Ex_spring_GPa': 'X-direction equivalent elastic modulus',
    'Ey_spring_GPa': 'Y-direction equivalent elastic modulus',
    'Ez_spring_GPa': 'Z-direction equivalent elastic modulus'
}
CN_TARGET_NAMES = TARGET_DISPLAY_NAMES

for i, col in enumerate(TARGET_COLS):
    true_i = y_true[:, i]
    pred_i = y_pred[:, i]

    fig_path = RESULT_DIR / f'true_vs_predicted_{col}.png'
    export_figure_data(
        pd.DataFrame({
            'target': col,
            'true_value': true_i,
            'predicted_value': pred_i
        }),
        fig_path
    )

    plt.figure(figsize=(5.5, 5))
    plt.scatter(true_i, pred_i, alpha=0.75)
    min_v = min(true_i.min(), pred_i.min())
    max_v = max(true_i.max(), pred_i.max())
    plt.plot([min_v, max_v], [min_v, max_v], linestyle='--')
    plt.xlabel('True value')
    plt.ylabel('Predicted value')
    plt.title(f'{TARGET_DISPLAY_NAMES[col]}: true vs. predicted')
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300)
    plt.show()


In [ ]:
# ============================================================
# 12.2 Residual distributions
# ============================================================

for i, col in enumerate(TARGET_COLS):
    residual = y_pred[:, i] - y_true[:, i]
    fig_path = RESULT_DIR / f'residual_distribution_{col}.png'
    export_figure_data(
        pd.DataFrame({
            'target': col,
            'true_value': y_true[:, i],
            'predicted_value': y_pred[:, i],
            'residual': residual
        }),
        fig_path,
        suffix='_raw_values.csv'
    )
    counts, bin_edges = np.histogram(residual, bins=30)
    export_figure_data(
        pd.DataFrame({
            'bin_left': bin_edges[:-1],
            'bin_right': bin_edges[1:],
            'count': counts
        }),
        fig_path,
        suffix='_histogram_bins.csv'
    )

    plt.figure(figsize=(6, 4))
    plt.hist(residual, bins=30)
    plt.xlabel('Prediction residual')
    plt.ylabel('Number of samples')
    plt.title(f'{TARGET_DISPLAY_NAMES[col]}: residual distribution')
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300)
    plt.show()


## 13. 可解释性分析：SHAP 代理解释（PM-GNN 预测结果）


In [ ]:
# ============================================================
# 13.1 SHAP：解释渗流模型的图级预测依据
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

EXPLAIN_DIR = RESULT_DIR / "explainability"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)

def make_graph_level_feature_table(graph_list):
    """
    将 PM-GNN 图数据转换成图级解释特征表。
    解释特征包括：
    1）graph 字典中已有的样本级结构信息；
    2）global_descriptors；
    3）节点特征统计；
    4）拓扑度数统计；
    5）如存在 edge_attr，则加入边特征统计。
    """
    rows = []
    graph_scalar_cols = [
        "porosity",
        "connected_porosity",
        "num_pores",
        "num_throats",
        "avg_pore_radius",
        "coordination_number",
    ]

    for g in graph_list:
        row = {
            "sample_id": g["sample_id"],
            "rock_id": g.get("rock_id", "unknown"),
            "split": g.get("split", "unknown"),
        }

        # 基础图中保留的图级标量信息。
        for name in graph_scalar_cols:
            val = g.get(name, np.nan)
            row[f"graph__{name}"] = float(val) if np.isfinite(val) else 0.0

        # global_descriptors：如果图对象中包含 global_features，则加入解释特征。
        if "global_features" in g:
            gf = np.asarray(g["global_features"], dtype=float).reshape(-1)
            gf_names = g.get("global_feature_names", GLOBAL_FEATURE_COLS if "GLOBAL_FEATURE_COLS" in globals() else [])
            for j, val in enumerate(gf):
                name = gf_names[j] if j < len(gf_names) else f"global_feature_{j}"
                row[f"global__{name}"] = float(val) if np.isfinite(val) else 0.0

        # 节点特征统计。
        x_raw = np.asarray(g["x"], dtype=float)
        for j, name in enumerate(NODE_FEATURE_COLS):
            vals = x_raw[:, j]
            row[f"node_mean__{name}"] = float(np.nanmean(vals))
            row[f"node_std__{name}"] = float(np.nanstd(vals))
            row[f"node_max__{name}"] = float(np.nanmax(vals))
            row[f"node_min__{name}"] = float(np.nanmin(vals))

        # 拓扑度数统计：从 edge_index 直接计算，不需要 edge_attr。
        edge_index = np.asarray(g["edge_index"], dtype=np.int64)
        n_nodes = int(x_raw.shape[0])
        if edge_index.size > 0 and n_nodes > 0:
            src = edge_index[:, 0]
            deg = np.bincount(src[(src >= 0) & (src < n_nodes)], minlength=n_nodes).astype(float)
            row["topology__mean_degree"] = float(np.nanmean(deg))
            row["topology__std_degree"] = float(np.nanstd(deg))
            row["topology__max_degree"] = float(np.nanmax(deg))
            row["topology__edge_density"] = float(edge_index.shape[0] / max(n_nodes * (n_nodes - 1), 1))
        else:
            row["topology__mean_degree"] = 0.0
            row["topology__std_degree"] = 0.0
            row["topology__max_degree"] = 0.0
            row["topology__edge_density"] = 0.0

        # 兼容：如果后续某个图对象额外带有 edge_attr，则也自动加入边统计；
        # 如果没有，则完全跳过，不影响基础图运行。
        if "edge_attr" in g and "EDGE_FEATURE_COLS" in globals() and len(EDGE_FEATURE_COLS) > 0:
            e_raw = np.asarray(g["edge_attr"], dtype=float)
            if e_raw.ndim == 2 and e_raw.shape[1] == len(EDGE_FEATURE_COLS):
                for j, name in enumerate(EDGE_FEATURE_COLS):
                    vals = e_raw[:, j]
                    row[f"edge_mean__{name}"] = float(np.nanmean(vals))
                    row[f"edge_std__{name}"] = float(np.nanstd(vals))
                    row[f"edge_max__{name}"] = float(np.nanmax(vals))

        rows.append(row)

    df = pd.DataFrame(rows)
    feature_cols = [c for c in df.columns if c not in ["sample_id", "rock_id", "split"]]
    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df, feature_cols


def predict_perm_scaled_for_graphs(graph_list, batch_size=32):
    preds = []
    for i in range(0, len(graph_list), batch_size):
        batch = graph_list[i:i+batch_size]
        x, edge_index, edge_attr, y, batch_index = batch_graphs(batch, task='perm')
        pred = perm_model(x, edge_index, edge_attr, batch_index, training=False).numpy()
        preds.append(pred)
    return np.vstack(preds)


graph_feature_df, shap_feature_cols = make_graph_level_feature_table(graphs)
graph_to_pos = {g["sample_id"]: i for i, g in enumerate(graphs)}

perm_pred_scaled_all = predict_perm_scaled_for_graphs(graphs, batch_size=32)
for j, target in enumerate(PERM_TARGET_COLS):
    graph_feature_df[f"gnn_pred_scaled__{target}"] = perm_pred_scaled_all[:, j]

train_mask = graph_feature_df["split"].eq("train").values
val_mask = graph_feature_df["split"].eq("val").values

X_train_shap = graph_feature_df.loc[train_mask, shap_feature_cols].copy()
X_val_shap = graph_feature_df.loc[val_mask, shap_feature_cols].copy()

shap_available = False
try:
    import shap
    shap_available = True
except Exception as e:
    print("未检测到 shap 包，将自动改用 permutation importance。需要 SHAP 图时请先运行：pip install shap")
    print("导入 shap 失败原因:", repr(e))

shap_summary_rows = []

for target in PERM_TARGET_COLS:
    y_train_sur = graph_feature_df.loc[train_mask, f"gnn_pred_scaled__{target}"].values
    y_val_sur = graph_feature_df.loc[val_mask, f"gnn_pred_scaled__{target}"].values

    surrogate = RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        min_samples_leaf=2,
        n_jobs=-1
    )
    surrogate.fit(X_train_shap, y_train_sur)
    val_sur_pred = surrogate.predict(X_val_shap)
    fidelity_r2 = r2_score(y_val_sur, val_sur_pred) if len(y_val_sur) > 1 else np.nan
    print(f"{target} 的 SHAP 代理模型拟合度 R²: {fidelity_r2:.4f}")

    if shap_available:
        explainer = shap.TreeExplainer(surrogate)
        shap_values = explainer.shap_values(X_val_shap)

        mean_abs = np.mean(np.abs(shap_values), axis=0)
        imp_df = pd.DataFrame({
            "feature": shap_feature_cols,
            "mean_abs_shap": mean_abs,
            "target": target,
            "surrogate_fidelity_r2": fidelity_r2
        }).sort_values("mean_abs_shap", ascending=False)
        imp_df.to_csv(EXPLAIN_DIR / f"SHAP重要性_{target}.csv", index=False, encoding="utf-8-sig")
        shap_summary_fig_path = EXPLAIN_DIR / f"SHAP_summary_{target}.png"
        shap_bar_fig_path = EXPLAIN_DIR / f"SHAP_bar_{target}.png"
        shap_values_df = pd.DataFrame(shap_values, columns=shap_feature_cols)
        shap_values_df.insert(0, 'sample_id', graph_feature_df.loc[val_mask, 'sample_id'].values)
        feature_values_df = X_val_shap.copy()
        feature_values_df.insert(0, 'sample_id', graph_feature_df.loc[val_mask, 'sample_id'].values)
        export_figure_data(shap_values_df, shap_summary_fig_path, suffix='_shap_values.csv')
        export_figure_data(feature_values_df, shap_summary_fig_path, suffix='_feature_values.csv')
        shap_summary_rows.append(imp_df.head(20))

        plt.figure(figsize=(8, 6))
        shap.summary_plot(
            shap_values,
            X_val_shap,
            feature_names=shap_feature_cols,
            max_display=20,
            show=False
        )
        plt.title(f"SHAP summary：{target}")
        plt.tight_layout()
        plt.savefig(shap_summary_fig_path, dpi=300, bbox_inches="tight")
        plt.show()

        plt.figure(figsize=(8, 5))
        top_imp = imp_df.head(20).iloc[::-1]
        plt.barh(top_imp["feature"], top_imp["mean_abs_shap"])
        plt.xlabel("mean(|SHAP value|)")
        plt.title(f"SHAP feature importance: {target}")
        plt.tight_layout()
        export_figure_data(top_imp, shap_bar_fig_path)
        plt.savefig(shap_bar_fig_path, dpi=300, bbox_inches="tight")
        plt.show()
    else:
        perm = permutation_importance(
            surrogate,
            X_val_shap,
            y_val_sur,
            n_repeats=20,
            random_state=SEED,
            scoring="r2"
        )
        imp_df = pd.DataFrame({
            "feature": shap_feature_cols,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
            "target": target,
            "surrogate_fidelity_r2": fidelity_r2
        }).sort_values("importance_mean", ascending=False)
        imp_df.to_csv(EXPLAIN_DIR / f"Permutation重要性_{target}.csv", index=False, encoding="utf-8-sig")
        shap_summary_rows.append(imp_df.head(20))

if shap_summary_rows:
    shap_top_all = pd.concat(shap_summary_rows, ignore_index=True)
    shap_top_all.to_csv(EXPLAIN_DIR / "渗流预测解释性特征重要性汇总.csv", index=False, encoding="utf-8-sig")
    display(shap_top_all.head(30))

print("SHAP/特征重要性结果目录:", EXPLAIN_DIR)


### 13.2 SHAP：解释力学模型的图级预测依据

上一节解释的是渗流模型 `kx/ky/kz`。本节沿用同一套图级结构特征表，额外解释力学模型 `Ex/Ey/Ez` 的 GNN 预测结果。为了保持运行速度，仍使用随机森林代理模型拟合 GNN 输出，再用 SHAP 或 permutation importance 计算特征贡献。

In [ ]:
# ============================================================
# 13.2 SHAP：解释力学模型的图级预测依据
# ============================================================
# 说明：
# 1. 这里不改动已经训练好的 mech_model，只在解释性分析阶段新增力学 SHAP。
# 2. 采用与渗流 SHAP 相同的“图级结构特征代理模型”思路：
#    图级结构特征 -> 随机森林代理模型 -> 拟合 GNN 的力学预测输出 -> SHAP解释。
# 3. 这样比直接对变节点数 GNN 做 DeepExplainer 更稳定、更快，也更适合论文补充解释。

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

EXPLAIN_DIR = RESULT_DIR / "explainability"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)

def predict_mech_scaled_for_graphs(graph_list, batch_size=32):
    """批量获得力学 GNN 模型在标准化空间中的预测值。"""
    preds = []
    for i in range(0, len(graph_list), batch_size):
        batch = graph_list[i:i+batch_size]
        x, edge_index, edge_attr, y, batch_index = batch_graphs(batch, task='mech')
        pred = mech_model(x, edge_index, edge_attr, batch_index, training=False).numpy()
        preds.append(pred)
    return np.vstack(preds)


# 如果已经运行过 13.1，则直接复用 graph_feature_df 和 shap_feature_cols；
# 如果单独运行本节，则自动重新生成图级特征表。
if "graph_feature_df" not in globals() or "shap_feature_cols" not in globals():
    graph_feature_df, shap_feature_cols = make_graph_level_feature_table(graphs)

# 将力学 GNN 的预测结果加入图级解释表。
mech_pred_scaled_all = predict_mech_scaled_for_graphs(graphs, batch_size=32)
for j, target in enumerate(MECH_TARGET_COLS):
    graph_feature_df[f"gnn_pred_scaled__{target}"] = mech_pred_scaled_all[:, j]

train_mask = graph_feature_df["split"].eq("train").values
val_mask = graph_feature_df["split"].eq("val").values

X_train_mech_shap = graph_feature_df.loc[train_mask, shap_feature_cols].copy()
X_val_mech_shap = graph_feature_df.loc[val_mask, shap_feature_cols].copy()

# 保持和渗流 SHAP 一致：优先用 SHAP；如果环境未安装 shap，则自动退化为 permutation importance。
mech_shap_available = False
try:
    import shap
    mech_shap_available = True
except Exception as e:
    print("未检测到 shap 包，将自动改用 permutation importance。需要 SHAP 图时请先运行：pip install shap")
    print("导入 shap 失败原因:", repr(e))

MECH_SHAP_SURROGATE_TREES = 200
MECH_SHAP_MAX_DISPLAY = 20

mech_shap_summary_rows = []

for target in MECH_TARGET_COLS:
    y_train_sur = graph_feature_df.loc[train_mask, f"gnn_pred_scaled__{target}"].values
    y_val_sur = graph_feature_df.loc[val_mask, f"gnn_pred_scaled__{target}"].values

    mech_surrogate = RandomForestRegressor(
        n_estimators=MECH_SHAP_SURROGATE_TREES,
        random_state=SEED,
        min_samples_leaf=2,
        n_jobs=-1
    )
    mech_surrogate.fit(X_train_mech_shap, y_train_sur)
    val_sur_pred = mech_surrogate.predict(X_val_mech_shap)
    fidelity_r2 = r2_score(y_val_sur, val_sur_pred) if len(y_val_sur) > 1 else np.nan
    print(f"{target} 的力学 SHAP 代理模型拟合度 R²: {fidelity_r2:.4f}")

    if mech_shap_available:
        mech_explainer = shap.TreeExplainer(mech_surrogate)
        mech_shap_values = mech_explainer.shap_values(X_val_mech_shap)

        mean_abs = np.mean(np.abs(mech_shap_values), axis=0)
        imp_df = pd.DataFrame({
            "feature": shap_feature_cols,
            "mean_abs_shap": mean_abs,
            "target": target,
            "task": "mechanics",
            "surrogate_fidelity_r2": fidelity_r2
        }).sort_values("mean_abs_shap", ascending=False)

        imp_df.to_csv(EXPLAIN_DIR / f"SHAP重要性_力学_{target}.csv", index=False, encoding="utf-8-sig")
        mech_shap_summary_fig_path = EXPLAIN_DIR / f"SHAP_summary_mechanics_{target}.png"
        mech_shap_bar_fig_path = EXPLAIN_DIR / f"SHAP_bar_mechanics_{target}.png"
        mech_shap_values_df = pd.DataFrame(mech_shap_values, columns=shap_feature_cols)
        mech_shap_values_df.insert(0, 'sample_id', graph_feature_df.loc[val_mask, 'sample_id'].values)
        mech_feature_values_df = X_val_mech_shap.copy()
        mech_feature_values_df.insert(0, 'sample_id', graph_feature_df.loc[val_mask, 'sample_id'].values)
        export_figure_data(mech_shap_values_df, mech_shap_summary_fig_path, suffix='_shap_values.csv')
        export_figure_data(mech_feature_values_df, mech_shap_summary_fig_path, suffix='_feature_values.csv')
        mech_shap_summary_rows.append(imp_df.head(MECH_SHAP_MAX_DISPLAY))

        plt.figure(figsize=(8, 6))
        shap.summary_plot(
            mech_shap_values,
            X_val_mech_shap,
            feature_names=shap_feature_cols,
            max_display=MECH_SHAP_MAX_DISPLAY,
            show=False
        )
        plt.title(f"SHAP summary: mechanics model {target}")
        plt.tight_layout()
        plt.savefig(mech_shap_summary_fig_path, dpi=300, bbox_inches="tight")
        plt.show()

        plt.figure(figsize=(8, 5))
        top_imp = imp_df.head(MECH_SHAP_MAX_DISPLAY).iloc[::-1]
        plt.barh(top_imp["feature"], top_imp["mean_abs_shap"])
        plt.xlabel("mean(|SHAP value|)")
        plt.title(f"SHAP feature importance: mechanics model {target}")
        plt.tight_layout()
        export_figure_data(top_imp, mech_shap_bar_fig_path)
        plt.savefig(mech_shap_bar_fig_path, dpi=300, bbox_inches="tight")
        plt.show()

    else:
        perm = permutation_importance(
            mech_surrogate,
            X_val_mech_shap,
            y_val_sur,
            n_repeats=20,
            random_state=SEED,
            scoring="r2"
        )
        imp_df = pd.DataFrame({
            "feature": shap_feature_cols,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
            "target": target,
            "task": "mechanics",
            "surrogate_fidelity_r2": fidelity_r2
        }).sort_values("importance_mean", ascending=False)

        imp_df.to_csv(EXPLAIN_DIR / f"Permutation重要性_力学_{target}.csv", index=False, encoding="utf-8-sig")
        mech_shap_summary_rows.append(imp_df.head(MECH_SHAP_MAX_DISPLAY))

if mech_shap_summary_rows:
    mech_shap_top_all = pd.concat(mech_shap_summary_rows, ignore_index=True)
    mech_shap_top_all.to_csv(
        EXPLAIN_DIR / "力学预测解释性特征重要性汇总.csv",
        index=False,
        encoding="utf-8-sig"
    )
    display(mech_shap_top_all.head(30))

print("力学 SHAP/特征重要性结果目录:", EXPLAIN_DIR)


In [ ]:
# ============================================================
# 13.3 注意力图说明
# ============================================================
# 本 notebook 是“基础 GNN 对照版”，模型结构中没有 attention pooling，也没有 edge gate。
# 因此本节不再绘制“节点注意力图/边门控权重图”。
#
# 如果后续需要可解释性，可使用前面的：
# 1）13.1 渗流 SHAP 代理解释；
# 2）13.2 力学 SHAP 代理解释；
# 3）14 图级 embedding PCA 可视化。
#
# 注意：只有在增强模型中显式加入 att_mlp 或 edge_gate_mlp 后，
# 才建议绘制 attention / edge gate 图。
print("基础GNN没有attention/edge gate，本节跳过。请使用13.1/13.2的SHAP结果进行解释。")


## 14. PM-GNN 图级嵌入可视化


In [ ]:
# ============================================================
# 14. PM-GNN 批处理版 embedding 提取（安全访问所有属性）
# ============================================================

def get_embeddings_batch(model, graph_list, task, batch_size=32):
    embs = []
    props = []
    
    for i in tqdm(range(0, len(graph_list), batch_size), desc='提取Embedding'):
        batch = graph_list[i:i+batch_size]
        x, edge_index, edge_attr, y, batch_index = batch_graphs(batch, task=task)
        _, emb = model(x, edge_index, edge_attr, batch_index, training=False, return_embedding=True)
        embs.append(emb.numpy())
        
        # 安全访问所有属性：不存在的键返回NaN，不会报错
        for g in batch:
            props.append({
                # 基本信息（这些应该都存在）
                'sample_id': g['sample_id'],
                'rock_id': g.get('rock_id', 'unknown'),
                'rock_name': g.get('rock_name', g.get('rock_id', 'unknown')),
                'dataset_split': g.get('split', 'unknown'),
                
                # 孔隙结构属性（用get()安全访问）
                'porosity': g['porosity'],
                'connected_porosity': g['connected_porosity'],
                'num_pores': g.get('num_pores', np.nan),
                'num_throats': g.get('num_throats', np.nan),
                'avg_pore_radius': g.get('avg_pore_radius', np.nan),
                'avg_throat_radius': g.get('avg_throat_radius', np.nan),
                'coordination_number': g.get('coordination_number', np.nan),
                
                # 物理属性：优先用y_raw（原始值），没有就用y（标准化后的值）
                'kx_um2': g.get('y_raw', g['y'])[0],
                'ky_um2': g.get('y_raw', g['y'])[1],
                'kz_um2': g.get('y_raw', g['y'])[2],
                'Ex_spring_GPa': g.get('y_raw', g['y'])[3],
                'Ey_spring_GPa': g.get('y_raw', g['y'])[4],
                'Ez_spring_GPa': g.get('y_raw', g['y'])[5]
            })
    
    return np.vstack(embs), pd.DataFrame(props)


# 先运行这个调试函数，查看你的graph对象实际包含哪些字段
def inspect_graph_structure(graph):
    """查看单个graph对象的所有键和值类型"""
    print('Graph对象包含的所有键：')
    for key, value in graph.items():
        if isinstance(value, (int, float, str, bool)):
            print(f'  {key}: {value}')
        else:
            print(f'  {key}: {type(value)} (shape: {getattr(value, "shape", "N/A")})')


# 运行调试，查看第一个graph的结构
print('=== 查看graph对象结构 ===')
inspect_graph_structure(graphs[0])
print('\n' + '='*50 + '\n')


# 提取embedding
print('提取渗流模型Embedding:')
perm_embs, props_df = get_embeddings_batch(perm_model, graphs, task='perm')

print('\n提取力学模型Embedding:')
mech_embs, _ = get_embeddings_batch(mech_model, graphs, task='mech')

# 查看props_df的列名，确认哪些属性成功提取了
print('\n成功提取的属性：')
print(props_df.columns.tolist())

# 查看缺失值统计
print('\n属性缺失值统计：')
print(props_df.isnull().sum())


# PCA降维
from sklearn.decomposition import PCA
pca_perm = PCA(n_components=2, random_state=SEED)
pca_mech = PCA(n_components=2, random_state=SEED)
perm_emb2 = pca_perm.fit_transform(perm_embs)
mech_emb2 = pca_mech.fit_transform(mech_embs)

print(f'\n渗流模型PCA解释方差比：PC1={pca_perm.explained_variance_ratio_[0]:.2%}, PC2={pca_perm.explained_variance_ratio_[1]:.2%}')
print(f'力学模型PCA解释方差比：PC1={pca_mech.explained_variance_ratio_[0]:.2%}, PC2={pca_mech.explained_variance_ratio_[1]:.2%}')


# ====================== 自动过滤不存在的属性 ======================
# 只保留实际存在且没有全部缺失的属性
available_cols = props_df.columns[props_df.notnull().any()].tolist()

# 定义你想要可视化的属性和对应的中文名称
visualization_config = {
    'porosity': 'Total porosity',
    'connected_porosity': 'Connected porosity',
    'num_pores': 'Number of pores',
    'num_throats': 'Number of throats',
    'avg_pore_radius': 'Mean pore radius',
    'coordination_number': 'Mean coordination number',
    'kx_um2': 'X-direction permeability',
    'ky_um2': 'Y-direction permeability',
    'kz_um2': 'Z-direction permeability',
    'Ex_spring_GPa': 'X-direction elastic modulus',
    'Ey_spring_GPa': 'Y-direction elastic modulus',
    'Ez_spring_GPa': 'Z-direction elastic modulus'
}

# 生成可视化列表（自动过滤不存在的属性）
visualization_list = []

# 渗流模型可视化
print('\n=== 渗流模型特征可视化 ===')
for col, cname in visualization_config.items():
    if col in available_cols:
        visualization_list.append((col, cname, perm_emb2, pca_perm))

# 力学模型可视化（只添加弹性模量相关）
print('\n=== 力学模型特征可视化 ===')
for col in ['Ex_spring_GPa', 'Ey_spring_GPa', 'Ez_spring_GPa']:
    if col in available_cols:
        cname = visualization_config[col]
        visualization_list.append((col, cname + ' (mechanics model)', mech_emb2, pca_mech))


# 批量画图
for col, cname, emb, pca_used in visualization_list:
    plt.figure(figsize=(6, 5))
    sc = plt.scatter(
        emb[:, 0], emb[:, 1], 
        c=props_df[col].values, 
        alpha=0.8, 
        s=20,
        cmap='viridis'
    )
    plt.colorbar(sc, label=cname)
    plt.xlabel(f'First principal component (explained variance: {pca_used.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'Second principal component (explained variance: {pca_used.explained_variance_ratio_[1]:.1%})')
    plt.title(f'PCA of GNN graph-level embeddings colored by {cname}')
    plt.tight_layout()
    fig_path = RESULT_DIR / f'PCA_{col}.png'
    export_figure_data(
        pd.DataFrame({
            'sample_id': props_df['sample_id'].values,
            'PC1': emb[:, 0],
            'PC2': emb[:, 1],
            'color_variable': col,
            'color_variable_label': cname,
            'color_value': props_df[col].values
        }),
        fig_path
    )
    plt.savefig(fig_path, dpi=600)
    plt.show()

## 15. 保存 PM-GNN 模型


In [ ]:
# ============================================================
# 15. 保存改进后的 PM-GNN 模型与标准化器
# ============================================================

import joblib

perm_weights_path = RESULT_DIR / (
    "pore_topology_pm_gnn_flow_path_descriptors_perm_"
    f"{SPLIT_MODE}.weights.h5"
)
mech_weights_path = RESULT_DIR / (
    "pore_topology_pm_gnn_flow_path_descriptors_mech_"
    f"{SPLIT_MODE}.weights.h5"
)

perm_model.save_weights(perm_weights_path)
mech_model.save_weights(mech_weights_path)

joblib.dump(
    node_scaler,
    RESULT_DIR / "node_scaler.pkl"
)
joblib.dump(
    edge_scaler,
    RESULT_DIR / "edge_scaler.pkl"
)
joblib.dump(
    perm_global_feature_scaler,
    RESULT_DIR / "perm_global_feature_scaler.pkl"
)
joblib.dump(
    mech_global_feature_scaler,
    RESULT_DIR / "mech_global_feature_scaler.pkl"
)
joblib.dump(
    perm_y_scaler,
    RESULT_DIR / "perm_y_scaler.pkl"
)
joblib.dump(
    mech_y_scaler,
    RESULT_DIR / "mech_y_scaler.pkl"
)

pd.DataFrame({
    "perm_global_feature": perm_global_feature_cols
}).to_csv(
    RESULT_DIR / "perm_global_feature_names.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "mech_global_feature": mech_global_feature_cols
}).to_csv(
    RESULT_DIR / "mech_global_feature_names.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "flow_path_feature": FLOW_GLOBAL_FEATURE_COLS
}).to_csv(
    RESULT_DIR / "flow_path_descriptor_names.csv",
    index=False,
    encoding="utf-8-sig"
)

model_config = {
    "model_name": "PM-GNN-flow",
    "definition": (
        "GNN + global_descriptors + "
        "flow-specific path descriptors"
    ),
    "hidden_dim": PM_HIDDEN_DIM,
    "num_layers": PM_NUM_LAYERS,
    "dropout": PM_DROPOUT,
    "use_global_descriptors": (
        USE_GLOBAL_DESCRIPTORS
    ),
    "use_flow_path_descriptors": (
        USE_FLOW_PATH_DESCRIPTORS
    ),
    "use_direction_conditioned_perm_head": False,
    "use_enhanced_mp": USE_ENHANCED_MP,
    "node_feature_cols": NODE_FEATURE_COLS,
    "edge_feature_cols": EDGE_FEATURE_COLS,
    "base_global_feature_cols": (
        BASE_GLOBAL_FEATURE_COLS
    ),
    "flow_global_feature_cols": (
        FLOW_GLOBAL_FEATURE_COLS
    ),
    "perm_global_feature_cols": (
        perm_global_feature_cols
    ),
    "mech_global_feature_cols": (
        mech_global_feature_cols
    ),
    "target_cols": TARGET_COLS,
    "split_mode": SPLIT_MODE,
}

with open(
    RESULT_DIR / "pm_gnn_flow_model_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print("渗流模型权重已保存:", perm_weights_path)
print("力学模型权重已保存:", mech_weights_path)
print(
    "渗流 global scaler 已保存:",
    RESULT_DIR / "perm_global_feature_scaler.pkl"
)
print(
    "力学 global scaler 已保存:",
    RESULT_DIR / "mech_global_feature_scaler.pkl"
)
print(
    "渗流路径描述符名称已保存:",
    RESULT_DIR / "flow_path_descriptor_names.csv"
)


## 16. 模型定位

当前模型以原始 `GNN + global_descriptors` 为基础，仅增强 global_descriptors：

- 增加 x/y/z 方向的贯通路径、骨架比例、迂曲度、最优路径导通能力和瓶颈描述符；
- 保持原 BasicGraphConv、mean pooling 和三输出回归头不变；
- flow descriptors 只进入渗流模型，力学模型保持原输入体系。

该版本用于单独判断“渗流路径描述符”是否能提升 `kx、ky、kz`。
